# English → Russian translation: the complete experiment

    This notebook embeds all pipeline code, including the custom Transformer and Marian Trainer.
    It has **no project imports**. The repository checkout is used only to read the attributed raw data.
    Training is complete: **all training flags default to False**, so Run all loads existing Drive results.
    Enable flags only for optional reproduction; a T4 GPU is required for full-size model runs.
    Final measured training: fast 7.9 min, large 74.1 min, Helsinki fine-tuning 31.7 min, plus evaluation/setup.

    ## 1. Install the pinned environment
    Run installation before importing torch/transformers. In a reused session, restart the runtime if those
    packages were already imported with different versions. CPU smoke mode skips installation and uses tiny random models;
    it is an execution check, not pretrained Helsinki validation.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path
CPU_SMOKE = os.environ.get('EN_RU_NOTEBOOK_SMOKE') == '1'
def stream(command):
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        status = process.wait()
    if status:
        raise subprocess.CalledProcessError(status, command)
PINNED_PACKAGES = ['torch==2.6.0', 'sentencepiece==0.2.0', 'sacrebleu==2.5.1', 'PyYAML==6.0.2', 'numpy==2.2.3', 'pytest==8.3.5', 'nbformat==5.10.4', 'tqdm==4.67.1', 'matplotlib==3.10.1', 'transformers==4.49.0', 'accelerate==1.4.0', 'huggingface-hub==0.29.3', 'tokenizers==0.21.1', 'safetensors==0.5.3', 'sacremoses==0.1.1', 'torchvision==0.21.0']
if not CPU_SMOKE:
    stream([sys.executable, '-m', 'pip', 'install', *PINNED_PACKAGES])

## 2. Drive, run flags and data location
    Existing runs are read from the paths supplied with the final Colab results. No training runs by default.
    New optional training writes distinct `*_full_notebook` directories and refuses to overwrite an existing run.
    Best-only checkpoints omit optimizer/resume state; an interrupted reproduction starts a fresh run.
    Overwriting/replacing a Drive checkpoint can put previous versions in **Drive Trash**. Keeping one visible
    checkpoint does not clear Trash; inspect and empty it separately when reclaiming space.

In [ ]:
import copy
import csv
import hashlib
import json
import time
import torch
import yaml
RUN_FAST = False
RUN_LARGE = False
RUN_HELSINKI_ZERO_SHOT = False
RUN_HELSINKI_FT = False
RUN_EXAMPLE_INFERENCE = False  # saved weights only; never training
if CPU_SMOKE:
    RUN_FAST = RUN_LARGE = RUN_HELSINKI_ZERO_SHOT = RUN_HELSINKI_FT = True
    ROOT = Path(os.environ['EN_RU_NOTEBOOK_OUTPUT']).resolve()
    WORK = ROOT/'work'
    RAW = Path(os.environ['EN_RU_NOTEBOOK_RAW']).resolve()
else:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/en_ru_translation')
    WORK = Path('/content/en_ru_translation_full')
    RAW = WORK/'repository/data/raw/rus.txt'
    REPO_REF = 'feat/scratch-transformer'
    if not RAW.exists():
        WORK.mkdir(parents=True, exist_ok=True)
        stream(['git', 'clone', '--depth', '1', '--branch', REPO_REF,
                'https://github.com/vcdx1111/Machine-translation-English-to-Russian-.git', str(WORK/'repository')])
WORK.mkdir(parents=True, exist_ok=True)
assert RAW.is_file(), RAW
if any((RUN_FAST, RUN_LARGE, RUN_HELSINKI_ZERO_SHOT, RUN_HELSINKI_FT)) and not CPU_SMOKE:
    assert torch.cuda.is_available(), 'Select a Colab T4 GPU runtime'
print('Data:', RAW, 'Device:', 'cuda' if torch.cuda.is_available() and not CPU_SMOKE else 'cpu')
print('All training is disabled by default; existing Drive artifacts are reused.')

## 3. TSV loading, EDA, cleaning and leakage-resistant splitting
    The raw file has English, Russian and original contributor attribution. Inspect all rows, lengths,
    duplicates, missing values and multi-translation sources. Group normalized English variants before splitting;
    train-only tokenizer fitting occurs later. This is the same preprocessing as the modular pipeline.

In [ ]:
"""Audit UTF-8 TSV data and create deterministic source-grouped splits."""
import argparse
from collections import Counter, defaultdict
import hashlib
import json
from pathlib import Path
import random
import re
import unicodedata

def normalize(text):
    return ' '.join(unicodedata.normalize('NFC', text).split())

def group_key(text):
    return ''.join((c for c in normalize(text).casefold() if c.isalnum()))

def read_raw(path):
    rows, widths, malformed = ([], Counter(), 0)
    with open(path, encoding='utf-8-sig') as f:
        for number, line in enumerate(f, 1):
            columns = line.rstrip('\r\n').split('\t')
            widths[len(columns)] += 1
            if len(columns) != 3:
                malformed += 1
                continue
            rows.append({'en': columns[0], 'ru': columns[1], 'attribution': columns[2], 'line': number})
    return (rows, dict(widths), malformed)

def clean(rows, max_chars=300, max_words=60):
    result, seen, reasons = ([], set(), Counter())
    for row in rows:
        en, ru = (normalize(row['en']), normalize(row['ru']))
        if not en or not ru:
            reasons['empty'] += 1
        elif not re.search('[A-Za-z]', en) or not re.search('[А-Яа-яЁё]', ru):
            reasons['missing_expected_script'] += 1
        elif any((unicodedata.category(c) in ('Cc', 'Cs') and (not c.isspace()) for c in en + ru)):
            reasons['control_characters'] += 1
        elif max(len(en), len(ru)) > max_chars or max(len(en.split()), len(ru.split())) > max_words:
            reasons['length'] += 1
        elif (en, ru) in seen:
            reasons['duplicate_pair'] += 1
        else:
            seen.add((en, ru))
            result.append(dict(row, en=en, ru=ru))
    return (result, dict(reasons))

def split_rows(rows, seed=42):
    groups = defaultdict(list)
    for row in rows:
        groups[group_key(row['en'])].append(row)
    keys = sorted(groups)
    random.Random(seed).shuffle(keys)
    n = len(keys)
    cut1, cut2 = (int(n * 0.9), int(n * 0.95))
    return {name: [r for k in subset for r in groups[k]] for name, subset in zip(('train', 'val', 'test'), (keys[:cut1], keys[cut1:cut2], keys[cut2:]))}

def distribution(values):
    import numpy as np
    return {k: float(v) for k, v in zip(('min', 'p50', 'p90', 'p95', 'p99', 'max'), np.percentile(values, [0, 50, 90, 95, 99, 100]))}

def prepare(raw, out, seed=42, report=None):
    raw, out = (Path(raw), Path(out))
    rows, widths, malformed = read_raw(raw)
    if not rows:
        raise ValueError('No valid three-column TSV rows')
    cleaned, removed = clean(rows)
    splits = split_rows(cleaned, seed)
    if any((not v for v in splits.values())):
        raise ValueError('Not enough source groups for three nonempty splits')
    out.mkdir(parents=True, exist_ok=True)
    for name, values in splits.items():
        with (out / f'{name}.jsonl').open('w', encoding='utf-8') as f:
            for row in values:
                f.write(json.dumps(row, ensure_ascii=False) + '\n')
    translations = defaultdict(set)
    for row in rows:
        translations[normalize(row['en'])].add(normalize(row['ru']))
    audit = {'sha256': hashlib.sha256(raw.read_bytes()).hexdigest(), 'seed': seed, 'raw_lines': sum(widths.values()), 'column_counts': widths, 'malformed': malformed, 'missing': {k: sum((not r[k].strip() for r in rows)) for k in ('en', 'ru', 'attribution')}, 'noise_observations': {'non_nfc_rows': sum((any((unicodedata.normalize('NFC', r[k]) != r[k] for k in ('en', 'ru'))) for r in rows)), 'whitespace_normalized_rows': sum((any((normalize(r[k]) != unicodedata.normalize('NFC', r[k]) for k in ('en', 'ru'))) for r in rows)), 'identical_normalized_text': sum((normalize(r['en']).casefold() == normalize(r['ru']).casefold() for r in rows)), 'markup_like_rows': sum((bool(re.search('<[^>]+>|https?://', r['en'] + r['ru'])) for r in rows)), 'replacement_character_rows': sum(('�' in r['en'] + r['ru'] for r in rows))}, 'raw_duplicate_pairs': len(rows) - len({(r['en'], r['ru']) for r in rows}), 'sources_with_multiple_translations': sum((len(v) > 1 for v in translations.values())), 'lengths': {lang: {unit: distribution([len(r[lang]) if unit == 'characters' else len(r[lang].split()) for r in rows]) for unit in ('characters', 'words')} for lang in ('en', 'ru')}, 'removed': removed, 'retained': len(cleaned), 'split_rows': {k: len(v) for k, v in splits.items()}, 'split_groups': {k: len({group_key(r['en']) for r in v}) for k, v in splits.items()}, 'multitranslation_examples': [{'en': k, 'ru': sorted(v)[:5]} for k, v in translations.items() if len(v) > 1][:5]}
    (out / 'audit.json').write_text(json.dumps(audit, ensure_ascii=False, indent=2), encoding='utf-8')
    if report:
        Path(report).parent.mkdir(parents=True, exist_ok=True)
        Path(report).write_text('# Data report\n\nGenerated from the checked-in `data/raw/rus.txt`; no download is required.\n\nEach row has English, Russian, and Tatoeba attribution, separated by tabs. Attribution is retained in processed rows. Source: Kaggle “English-Russian dictionary for machine translate”, originally Tatoeba (CC-BY 2.0); the exact Kaggle listing URL was not supplied. Preserve the original attribution and review the dataset license before redistribution; rows identify CC-BY 2.0 (France).\n\n## Cleaning and leakage prevention\n\nNFC Unicode normalization and whitespace collapse; preserve case, punctuation, and Russian stress accents. Remove empty text, missing expected Latin/Cyrillic scripts, control characters, duplicate normalized pairs, and pairs over 300 characters or 60 whitespace words on either side. These are conservative heuristics, not semantic quality labels. No automatic language detector or manual quality certification is claimed.\n\nGroup by casefolded English alphanumeric characters (punctuation/whitespace removed). Shuffle sorted groups with seed 42; allocate 90/5/5 percent of groups. All translations and simple punctuation/case variants stay together. Paraphrases can still overlap; sentence-level splitting does not guarantee topic isolation. Fit SentencePiece on train only.\n\n## Measured statistics\n\n```json\n' + json.dumps(audit, ensure_ascii=False, indent=2) + '\n```\n', encoding='utf-8')
    return splits

def load_rows(directory, split):
    with (Path(directory) / f'{split}.jsonl').open(encoding='utf-8') as f:
        return [json.loads(line) for line in f]

def selected_splits(cfg):
    """The same cleaning, source-group split and optional CPU sampling for every model."""
    prepare(cfg['paths']['raw'], cfg['paths']['processed'], cfg['seed'])
    result = {}
    for name in ('train', 'val', 'test'):
        values = load_rows(cfg['paths']['processed'], name)
        random.Random(cfg['seed']).shuffle(values)
        limit = cfg['data'].get(f'max_{name}')
        result[name] = values[:limit] if limit else values
    return result

"""Shared result artifacts and exact held-out reference identity across model families."""
import hashlib
import json
from pathlib import Path

def test_identity(rows, data_hash, seed):
    pairs = sorted({(r['en'], r['ru']) for r in rows})
    payload = json.dumps(pairs, ensure_ascii=False, separators=(',', ':')).encode()
    return dict(schema_version=2, data_sha256=data_hash, split_seed=seed, test_split_sha256=hashlib.sha256(payload).hexdigest(), test_pairs=len(rows), test_sources=len({s for s, _ in pairs}), metric_policy='multi-reference sacreBLEU 13a + chrF2; exact English source grouping')

def write_artifacts(output, metrics, sources, references, predictions, samples_name='samples.txt'):
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    (output / 'metrics.json').write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding='utf-8')
    with (output / samples_name).open('w', encoding='utf-8') as f:
        for source, hyp in list(zip(sources, predictions))[:30]:
            f.write(f"EN: {source}\nREF: {' || '.join(references[source])}\nMODEL: {hyp}\n\n")

### Inspect the corpus and fix the split
    Full data is split before applying CPU-only subset limits. The final test identity is checked against
    the published hash; token caps never change the reference set. Raw lines are never silently sampled for EDA.

In [ ]:
ORIGINAL_CONFIGS = {'fast': {'seed': 42, 'paths': {'raw': 'data/raw/rus.txt', 'processed': 'data/processed/colab_fast', 'checkpoints': 'checkpoints/scratch_fast_v4', 'results': 'results/scratch_fast_v4'}, 'data': {'max_train': None, 'max_val': None, 'max_test': None}, 'tokenizer': {'vocab_size': 8000}, 'model': {'d_model': 256, 'heads': 4, 'layers': 3, 'd_ff': 1024, 'dropout': 0.1, 'max_length': 24, 'norm_first': False, 'tie_embeddings': False, 'initialization': 'legacy_xavier'}, 'training': {'device': 'cuda', 'cpu_threads': 2, 'max_tokens': 16384, 'max_sentences': 1024, 'bucket_size': 8192, 'num_workers': 2, 'pin_memory': True, 'prefetch_factor': 2, 'epochs': 12, 'warmup_steps': 800, 'label_smoothing': 0.1, 'gradient_clip': 1.0, 'mixed_precision': True, 'patience': 3, 'min_delta': 0.001, 'progress': True, 'log_every': 20, 'divergence': {'val_ratio': 1.25, 'val_delta': 0.5, 'max_consecutive_skips': 8, 'max_skip_fraction': 0.25, 'skip_check_steps': 20}}, 'evaluation': {'val_greedy_samples': 128, 'batch_size': 128, 'beam_size': 4, 'beam_batch_size': 64, 'beam_max_sources': None, 'length_penalty': 0.6, 'report_greedy': True}, 'experiment': 'scratch_fast_v4'}, 'large': {'seed': 42, 'paths': {'raw': 'data/raw/rus.txt', 'processed': 'data/processed/colab_large', 'checkpoints': 'checkpoints/scratch_large_stable_v4', 'results': 'results/scratch_large_stable_v4'}, 'data': {'max_train': None, 'max_val': None, 'max_test': None}, 'tokenizer': {'vocab_size': 16000}, 'model': {'d_model': 512, 'heads': 8, 'layers': 6, 'd_ff': 2048, 'dropout': 0.1, 'max_length': 64, 'norm_first': True, 'tie_embeddings': True, 'initialization': 'scaled_embedding'}, 'training': {'device': 'cuda', 'cpu_threads': 2, 'max_tokens': 16384, 'max_sentences': 1024, 'bucket_size': 8192, 'num_workers': 2, 'pin_memory': True, 'prefetch_factor': 2, 'epochs': 30, 'warmup_steps': 3000, 'lr_factor': 0.4, 'label_smoothing': 0.1, 'gradient_clip': 1.0, 'mixed_precision': True, 'patience': 5, 'min_delta': 0.001, 'progress': True, 'log_every': 10, 'divergence': {'val_ratio': 1.25, 'val_delta': 0.5, 'max_consecutive_skips': 8, 'max_skip_fraction': 0.25, 'skip_check_steps': 20}}, 'evaluation': {'val_greedy_samples': 128, 'batch_size': 128, 'beam_size': 4, 'beam_batch_size': 64, 'beam_max_sources': None, 'report_greedy': True, 'repetition_ablation_ngram': 3, 'length_penalty': 0.6}, 'experiment': 'scratch_large_stable_v4'}, 'helsinki': {'seed': 42, 'model': {'name': 'Helsinki-NLP/opus-mt-en-ru', 'revision': 'main'}, 'paths': {'raw': 'data/raw/rus.txt', 'processed': 'data/processed/helsinki', 'checkpoints': 'checkpoints/helsinki', 'results': 'results/helsinki'}, 'data': {'max_train': None, 'max_val': None, 'max_test': None, 'max_length': 64}, 'training': {'device': 'cuda', 'cpu_threads': 2, 'epochs': 3, 'batch_size': 64, 'eval_batch_size': 64, 'gradient_accumulation_steps': 2, 'learning_rate': 3e-05, 'warmup_ratio': 0.05, 'weight_decay': 0.01, 'label_smoothing': 0.1, 'gradient_clip': 1.0, 'mixed_precision': True, 'num_workers': 2, 'pin_memory': True, 'patience': 2, 'logging_steps': 25, 'progress': True}, 'evaluation': {'batch_size': 64, 'beam_batch_size': 32, 'beam_size': 4, 'length_penalty': 0.6, 'no_repeat_ngram_size': 0, 'max_new_tokens': 64}}}
FINAL_REPORT = {'provenance': 'User-reported Colab T4 results; original GPU artifacts remain in Google Drive. Not a reconstructed metrics.json or epoch log.', 'test_sources': 12556, 'test_split_sha256_prefix': 'bc6e2388', 'results_root': '/content/drive/MyDrive/en_ru_translation', 'runs': [{'name': 'Transformer fast v4', 'path': 'scratch_fast_v4_fullbeam/results', 'greedy_bleu': 54.53, 'greedy_chrf': 67.93, 'beam_bleu': 56.08, 'beam_chrf': 69.18, 'training_seconds': 475, 'training_status': 'completed'}, {'name': 'Transformer large v4', 'path': 'scratch_large_stable_v4/results', 'greedy_bleu': 58.65, 'greedy_chrf': 71.44, 'beam_bleu': 59.88, 'beam_chrf': 72.37, 'training_seconds': 4446, 'training_status': 'early_stopped'}, {'name': 'Helsinki zero-shot', 'path': 'helsinki_v1_seed42/results/zero_shot', 'greedy_bleu': 55.38, 'greedy_chrf': 71.78, 'beam_bleu': 58.1, 'beam_chrf': 73.09, 'training_seconds': 0, 'training_status': 'completed'}, {'name': 'Helsinki fine-tuned', 'path': 'helsinki_v1_seed42/results/finetuned', 'greedy_bleu': 56.41, 'greedy_chrf': 74.52, 'beam_bleu': 62.68, 'beam_chrf': 76.77, 'training_seconds': 1901, 'training_status': 'completed'}], 'repetition_ablation': {'name': 'Transformer large v4 + no-repeat-3', 'beam_bleu': 59.87, 'beam_chrf': 72.36}, 'test_split_sha256': 'bc6e2388f0938c3332e747f174c98d16530f69b4641f0cb6ddb3c074e8f39f30'}

FULL_ROWS = prepare(RAW, WORK/'processed', 42)
AUDIT = json.loads((WORK/'processed/audit.json').read_text())
print(json.dumps(AUDIT, ensure_ascii=False, indent=2))
FULL_IDENTITY = test_identity(FULL_ROWS['test'], AUDIT['sha256'], 42)
assert FULL_IDENTITY['test_split_sha256'] == FINAL_REPORT['test_split_sha256']
print('Shared held-out identity:', FULL_IDENTITY)
for name in ('train','val','test'):
    print(name, 'pairs:',len(FULL_ROWS[name]), 'first row:', FULL_ROWS[name][0])
assert not (set(group_key(r['en']) for r in FULL_ROWS['train']) & set(group_key(r['en']) for r in FULL_ROWS['test']))

## 4. Train-only joint SentencePiece BPE and token-budget batches
    Fast uses an 8k vocabulary and a 24-token cap; large uses 16k and 64. BOS/EOS count toward the cap.
    Length buckets limit padded source+target tokens to 16,384. Embedded notebook callables are not importable
    by spawned worker processes, so notebook DataLoaders use `num_workers=0`; CLI configs retain two workers.
    All model, schedule and batch-budget hyperparameters otherwise match the published configs.

In [ ]:
"""Joint SentencePiece BPE fitted exclusively to training pairs."""
from pathlib import Path
import sentencepiece as spm
PAD, UNK, BOS, EOS = (0, 1, 2, 3)

def train_tokenizer(rows, prefix, vocab_size):
    prefix = Path(prefix)
    prefix.parent.mkdir(parents=True, exist_ok=True)
    spm.SentencePieceTrainer.train(sentence_iterator=(r[lang] for r in rows for lang in ('en', 'ru')), model_prefix=str(prefix), vocab_size=vocab_size, model_type='bpe', pad_id=PAD, unk_id=UNK, bos_id=BOS, eos_id=EOS, character_coverage=1.0, hard_vocab_limit=False, shuffle_input_sentence=False, num_threads=1, minloglevel=2)
    return Tokenizer(str(prefix) + '.model')

class Tokenizer:

    def __init__(self, model):
        self.sp = spm.SentencePieceProcessor(model_file=str(model))

    def __len__(self):
        return self.sp.get_piece_size()

    def encode(self, text, max_length=128):
        return [BOS] + self.sp.encode(text, out_type=int)[:max_length - 2] + [EOS]

    def decode(self, ids):
        ids = list(ids)
        if EOS in ids:
            ids = ids[:ids.index(EOS)]
        return self.sp.decode([i for i in ids if i not in (PAD, BOS)])

"""Deterministic length buckets with a bound on padded source + target tokens."""
import random
import torch
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Sampler

def collate(batch):
    return tuple((pad_sequence([torch.as_tensor(x[i], dtype=torch.long) for x in batch], batch_first=True, padding_value=PAD) for i in (0, 1)))

class TokenBatchSampler(Sampler):

    def __init__(self, pairs, max_tokens, max_sentences=512, bucket_size=4096, shuffle=True, seed=42):
        self.lengths = [(len(s), len(t)) for s, t in pairs]
        self.max_tokens, self.max_sentences = (max_tokens, max_sentences)
        self.bucket_size, self.shuffle, self.seed = (bucket_size, shuffle, seed)
        if max_tokens < max((s + t for s, t in self.lengths), default=0):
            raise ValueError('max_tokens cannot fit the longest single pair')
        if max_sentences < 1 or bucket_size < 1:
            raise ValueError('max_sentences and bucket_size must be positive')
        self.set_epoch(0)

    def set_epoch(self, epoch):
        rng = random.Random(self.seed + epoch)
        indices = list(range(len(self.lengths)))
        if self.shuffle:
            rng.shuffle(indices)
        batches = []
        for offset in range(0, len(indices), self.bucket_size):
            pool = sorted(indices[offset:offset + self.bucket_size], key=lambda i: (max(self.lengths[i]), self.lengths[i]))
            batch, ms, mt = ([], 0, 0)
            for i in pool:
                s, t = self.lengths[i]
                ns, nt = (max(ms, s), max(mt, t))
                if batch and ((len(batch) + 1) * (ns + nt) > self.max_tokens or len(batch) >= self.max_sentences):
                    batches.append(batch)
                    batch, ms, mt = ([], 0, 0)
                batch.append(i)
                ms, mt = (max(ms, s), max(mt, t))
            if batch:
                batches.append(batch)
        if self.shuffle:
            rng.shuffle(batches)
        self.batches = batches

    def __iter__(self):
        return iter(self.batches)

    def __len__(self):
        return len(self.batches)

def make_loader(rows, tok, cfg, shuffle):
    pairs = [(tok.encode(r['en'], cfg['model']['max_length']), tok.encode(r['ru'], cfg['model']['max_length'])) for r in rows]
    training = cfg['training']
    workers = training.get('num_workers', 0)
    use_cuda = training['device'] != 'cpu' and torch.cuda.is_available()
    kwargs = dict(collate_fn=collate, num_workers=workers, pin_memory=training.get('pin_memory', False) and use_cuda, generator=torch.Generator().manual_seed(cfg['seed']))
    if workers:
        kwargs.update(persistent_workers=True, prefetch_factor=training.get('prefetch_factor', 2), multiprocessing_context='spawn')
    if training.get('max_tokens'):
        sampler = TokenBatchSampler(pairs, training['max_tokens'], training.get('max_sentences', 512), training.get('bucket_size', 4096), shuffle, cfg['seed'])
        return DataLoader(pairs, batch_sampler=sampler, **kwargs)
    return DataLoader(pairs, batch_size=training['batch_size'], shuffle=shuffle, **kwargs)

def set_loader_epoch(loader, epoch, seed):
    if isinstance(loader.batch_sampler, TokenBatchSampler):
        loader.batch_sampler.set_epoch(epoch)
    loader.generator.manual_seed(seed + epoch)

def loader_stats(loader):
    pairs = loader.dataset
    actual, padded = (0, 0)
    for batch in loader.batch_sampler:
        lengths = [(len(pairs[i][0]), len(pairs[i][1])) for i in batch]
        actual += sum((s + t for s, t in lengths))
        padded += len(batch) * (max((s for s, _ in lengths)) + max((t for _, t in lengths)))
    return dict(batches=len(loader), pairs=len(pairs), nonpad_tokens=actual, padded_tokens=padded, padding_fraction=1 - actual / padded, target_tokens=sum((len(t) - 1 for _, t in pairs)))

## 5. The Transformer, implemented directly in PyTorch
    Explicit multi-head scaled attention, sinusoidal positions, source padding and target causal masks,
    encoder/decoder residual blocks, Post-LN and Pre-LN. Large v4 ties input/output weights and normalizes
    embedding initialization before sqrt(d_model) scaling. No `nn.Transformer` is used.

In [ ]:
"""Configurable pre/post-norm Transformer with explicit scaled dot-product multi-head attention."""
import math
import torch
from torch import nn

def source_mask(tokens):
    return (tokens != PAD)[:, None, None, :]

def target_mask(tokens):
    n = tokens.size(1)
    return source_mask(tokens) & torch.ones(n, n, dtype=torch.bool, device=tokens.device).tril()[None, None]

class MultiHeadAttention(nn.Module):

    def __init__(self, d_model, heads, dropout):
        super().__init__()
        if d_model % heads:
            raise ValueError('d_model must be divisible by heads')
        self.heads, self.dk = (heads, d_model // heads)
        self.q, self.k, self.v, self.out = [nn.Linear(d_model, d_model) for _ in range(4)]
        self.dropout = nn.Dropout(dropout)

    def forward(self, query, key, value, mask):
        b = query.size(0)

        def project(layer, x):
            return layer(x).view(b, -1, self.heads, self.dk).transpose(1, 2)
        q, k, v = (project(self.q, query), project(self.k, key), project(self.v, value))
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.dk)
        weights = scores.masked_fill(~mask, torch.finfo(scores.dtype).min).softmax(-1)
        attended = self.dropout(weights) @ v
        return self.out(attended.transpose(1, 2).contiguous().view(b, -1, self.heads * self.dk))

class PositionalEncoding(nn.Module):

    def __init__(self, d_model, dropout, max_length):
        super().__init__()
        if d_model % 2:
            raise ValueError('d_model must be even')
        p = torch.arange(max_length).float()[:, None]
        rates = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000) / d_model))
        pe = torch.zeros(max_length, d_model)
        pe[:, 0::2], pe[:, 1::2] = (torch.sin(p * rates), torch.cos(p * rates))
        self.register_buffer('pe', pe[None])
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        if x.size(1) > self.pe.size(1):
            raise ValueError('Sequence exceeds configured positional encoding')
        return self.dropout(x + self.pe[:, :x.size(1)].to(x.dtype))

class EncoderLayer(nn.Module):

    def __init__(self, d_model, heads, d_ff, dropout, norm_first=False):
        super().__init__()
        self.norm_first = norm_first
        self.attn = MultiHeadAttention(d_model, heads, dropout)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Dropout(dropout), nn.Linear(d_ff, d_model))
        self.norm1, self.norm2 = (nn.LayerNorm(d_model), nn.LayerNorm(d_model))
        self.drop = nn.Dropout(dropout)

    def forward(self, x, mask):
        if self.norm_first:
            z = self.norm1(x)
            x = x + self.drop(self.attn(z, z, z, mask))
            return x + self.drop(self.ff(self.norm2(x)))
        x = self.norm1(x + self.drop(self.attn(x, x, x, mask)))
        return self.norm2(x + self.drop(self.ff(x)))

class DecoderLayer(nn.Module):

    def __init__(self, d_model, heads, d_ff, dropout, norm_first=False):
        super().__init__()
        self.norm_first = norm_first
        self.self_attn = MultiHeadAttention(d_model, heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, heads, dropout)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Dropout(dropout), nn.Linear(d_ff, d_model))
        self.norm1, self.norm2, self.norm3 = [nn.LayerNorm(d_model) for _ in range(3)]
        self.drop = nn.Dropout(dropout)

    def forward(self, x, memory, src_mask, tgt_mask):
        if self.norm_first:
            z = self.norm1(x)
            x = x + self.drop(self.self_attn(z, z, z, tgt_mask))
            x = x + self.drop(self.cross_attn(self.norm2(x), memory, memory, src_mask))
            return x + self.drop(self.ff(self.norm3(x)))
        x = self.norm1(x + self.drop(self.self_attn(x, x, x, tgt_mask)))
        x = self.norm2(x + self.drop(self.cross_attn(x, memory, memory, src_mask)))
        return self.norm3(x + self.drop(self.ff(x)))

class Transformer(nn.Module):

    def __init__(self, vocab_size, d_model=512, heads=8, layers=6, d_ff=2048, dropout=0.1, max_length=128, norm_first=False, tie_embeddings=False, initialization='legacy_xavier'):
        super().__init__()
        self.scale = math.sqrt(d_model)
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=PAD)
        self.position = PositionalEncoding(d_model, dropout, max_length)
        self.encoder = nn.ModuleList([EncoderLayer(d_model, heads, d_ff, dropout, norm_first) for _ in range(layers)])
        self.decoder = nn.ModuleList([DecoderLayer(d_model, heads, d_ff, dropout, norm_first) for _ in range(layers)])
        self.encoder_norm = nn.LayerNorm(d_model) if norm_first else nn.Identity()
        self.decoder_norm = nn.LayerNorm(d_model) if norm_first else nn.Identity()
        self.output = nn.Linear(d_model, vocab_size)
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)
        if initialization not in ('legacy_xavier', 'scaled_embedding'):
            raise ValueError('Unknown initialization')
        if initialization == 'scaled_embedding':
            nn.init.normal_(self.embedding.weight, mean=0.0, std=d_model ** (-0.5))
            for module in self.modules():
                if isinstance(module, nn.Linear) and module.bias is not None:
                    nn.init.zeros_(module.bias)
        if tie_embeddings:
            self.output.weight = self.embedding.weight
        with torch.no_grad():
            self.embedding.weight[PAD].zero_()

    def encode(self, src):
        x = self.position(self.embedding(src) * self.scale)
        mask = source_mask(src)
        for layer in self.encoder:
            x = layer(x, mask)
        return (self.encoder_norm(x), mask)

    def decode(self, tgt, memory, src_mask, last_only=False):
        x = self.position(self.embedding(tgt) * self.scale)
        mask = target_mask(tgt)
        for layer in self.decoder:
            x = layer(x, memory, src_mask, mask)
        return self.output(self.decoder_norm(x[:, -1:] if last_only else x))

    def forward(self, src, tgt):
        memory, mask = self.encode(src)
        return self.decode(tgt, memory, mask)

## 6. Batched greedy/beam decoding and shared multi-reference metrics
    Beam size 4 and length penalty 0.6 are used on the **whole** test set. Greedy is recorded separately.
    The large no-repeat-3 ablation remains separate from the primary score. sacreBLEU 13a and chrF2 share
    exactly the same references with Marian. CPU scores are labelled and must not be read as model quality.

In [ ]:
"""Vectorized greedy/beam decoding across sentences and beams."""
import argparse
from contextlib import nullcontext
import torch
from torch.nn.utils.rnn import pad_sequence
from tqdm.auto import tqdm

def block_repeated_ngrams(logits, sequences, n):
    """Mask repeated ngrams in parallel; n=0 leaves logits unchanged."""
    if n < 0:
        raise ValueError('no_repeat_ngram_size must be nonnegative')
    if n == 0 or sequences.size(1) < n:
        return logits
    windows = sequences.unfold(1, n, 1)
    matches = (windows[:, :, :-1] == sequences[:, None, -(n - 1):]).all(-1) if n > 1 else torch.ones(windows.shape[:2], dtype=torch.bool, device=sequences.device)
    counts = torch.zeros_like(logits, dtype=torch.int32)
    counts.scatter_add_(1, windows[:, :, -1], matches.int())
    return logits.masked_fill(counts > 0, -torch.inf)

@torch.inference_mode()
def batch_decode_ids(model, src, max_length=128, beam_size=1, length_penalty=0.6, mixed_precision=False, no_repeat_ngram_size=0):
    if beam_size < 1:
        raise ValueError('beam_size must be positive')
    model.eval()
    amp = torch.autocast('cuda', dtype=torch.float16) if mixed_precision and src.is_cuda else nullcontext()
    with amp:
        memory, mask = model.encode(src)
        batch, device = (src.size(0), src.device)
        memory = memory.repeat_interleave(beam_size, 0)
        if mask is not None:
            mask = mask.repeat_interleave(beam_size, 0)
        tokens = torch.full((batch, beam_size, 1), BOS, dtype=torch.long, device=device)
        scores = torch.full((batch, beam_size), -torch.inf, device=device)
        scores[:, 0] = 0
        lengths = torch.zeros(batch, beam_size, dtype=torch.long, device=device)
        finished = torch.zeros(batch, beam_size, dtype=torch.bool, device=device)
        for _ in range(max_length - 1):
            logits = model.decode(tokens.reshape(batch * beam_size, -1), memory, mask, last_only=True)[:, -1].float()
            logits = block_repeated_ngrams(logits, tokens.reshape(batch * beam_size, -1), no_repeat_ngram_size)
            logits[:, [PAD, BOS]] = -torch.inf
            logp = logits.log_softmax(-1).view(batch, beam_size, -1)
            vocab = logp.size(-1)
            if beam_size > vocab - 2:
                raise ValueError('beam_size exceeds available output tokens')
            logp = logp.masked_fill(finished[:, :, None], -torch.inf)
            logp[:, :, PAD] = torch.where(finished, 0.0, logp[:, :, PAD])
            candidate_scores = scores[:, :, None] + logp
            candidate_lengths = lengths + (~finished).long()
            penalty = ((5 + candidate_lengths.float()) / 6).pow(length_penalty)
            ranks = candidate_scores / penalty[:, :, None]
            _, indices = ranks.flatten(1).topk(beam_size, dim=-1)
            parent, word = (indices // vocab, indices % vocab)
            tokens = torch.cat((tokens.gather(1, parent[:, :, None].expand(-1, -1, tokens.size(-1))), word[:, :, None]), -1)
            scores = candidate_scores.flatten(1).gather(1, indices)
            lengths = candidate_lengths.gather(1, parent)
            finished = finished.gather(1, parent) | word.eq(EOS)
            if finished.all():
                break
        ranks = scores / ((5 + lengths.float()) / 6).pow(length_penalty)
        best = ranks.argmax(-1)
        return tokens[torch.arange(batch, device=device), best].tolist()

def decode_ids(model, src, max_length=128, beam_size=1, length_penalty=0.6):
    if src.size(0) != 1:
        raise ValueError('Use batch_decode_ids for multiple sources')
    return batch_decode_ids(model, src, max_length, beam_size, length_penalty)[0]

def translate_many(model, tokenizer, texts, device, max_length, beam_size=1, length_penalty=0.6, batch_size=64, mixed_precision=False, progress=True, no_repeat_ngram_size=0):
    encoded = [tokenizer.encode(s, max_length) for s in texts]
    order = sorted(range(len(texts)), key=lambda i: len(encoded[i]))
    predictions = [None] * len(texts)
    for start in tqdm(range(0, len(order), batch_size), desc=f'decode beam={beam_size}', disable=not progress, mininterval=1):
        indices = order[start:start + batch_size]
        src = pad_sequence([torch.tensor(encoded[i]) for i in indices], batch_first=True, padding_value=PAD).to(device)
        outputs = batch_decode_ids(model, src, max_length, beam_size, length_penalty, mixed_precision, no_repeat_ngram_size)
        for i, ids in zip(indices, outputs):
            predictions[i] = tokenizer.decode(ids)
    return predictions

def translate(model, tokenizer, text, device, max_length, beam_size=1, length_penalty=0.6):
    return translate_many(model, tokenizer, [text], device, max_length, beam_size, length_penalty, progress=False)[0]

"""Batched corpus evaluation; full greedy scores and a separately labelled beam subset."""
import argparse
from collections import defaultdict
import hashlib
import json
from pathlib import Path
import time
import torch
from sacrebleu.metrics import BLEU, CHRF

def load_checkpoint(path):
    state = torch.load(path, map_location='cpu', weights_only=False)
    cfg = state['config']
    import sentencepiece as spm
    tok = Tokenizer.__new__(Tokenizer)
    tok.sp = spm.SentencePieceProcessor(model_proto=state['tokenizer_model'])
    torch.set_num_threads(cfg['training'].get('cpu_threads', 2))
    device = torch.device('cuda' if torch.cuda.is_available() and cfg['training']['device'] != 'cpu' else 'cpu')
    model = Transformer(len(tok), **cfg['model']).to(device)
    model.load_state_dict(state['best_model'])
    return (model, tok, cfg, state, device)

def reference_groups(rows):
    references = defaultdict(list)
    for r in rows:
        if r['ru'] not in references[r['en']]:
            references[r['en']].append(r['ru'])
    return references

def score_predictions(predictions, sources, references):
    refs = [[references[s][i] if i < len(references[s]) else None for s in sources] for i in range(max((len(references[s]) for s in sources)))]
    bleu, chrf = (BLEU(tokenize='13a'), CHRF())
    b, c = (bleu.corpus_score(predictions, refs), chrf.corpus_score(predictions, refs))
    return dict(bleu=b.score, chrf=c.score, bleu_signature=str(bleu.get_signature()), chrf_signature=str(chrf.get_signature()))

def decode_sources(model, tok, cfg, sources, device, beam_size=1, batch_size=None, no_repeat_ngram_size=0):
    ev = cfg['evaluation']
    return translate_many(model, tok, sources, device, cfg['model']['max_length'], beam_size, ev['length_penalty'], batch_size or ev.get('batch_size', 64), cfg['training']['mixed_precision'], cfg['training'].get('progress', True), no_repeat_ngram_size)

def validation_greedy(model, tok, cfg, rows, device):
    refs = reference_groups(rows)
    n = cfg['evaluation'].get('val_greedy_samples', 0)
    sources = list(refs)[:n]
    if not sources:
        return dict(val_greedy_bleu=None, val_greedy_chrf=None, val_greedy_seconds=0.0, val_greedy_sources=0)
    begin = time.perf_counter()
    pred = decode_sources(model, tok, cfg, sources, device, 1)
    score = score_predictions(pred, sources, refs)
    return dict(val_greedy_bleu=score['bleu'], val_greedy_chrf=score['chrf'], val_greedy_seconds=time.perf_counter() - begin, val_greedy_sources=len(sources))

def evaluate_model(model, tok, cfg, rows, output, device, training_seconds, data_hash):
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    references = reference_groups(rows)
    sources = list(references)
    if not sources:
        raise ValueError('Empty test split')
    begin = time.perf_counter()
    ev = cfg['evaluation']
    beam = ev['beam_size']
    limit = ev.get('beam_max_sources')
    if beam < 1 or (limit is not None and limit < 0):
        raise ValueError('beam_size must be positive and beam_max_sources must be nonnegative')
    full_beam = beam > 1 and (limit is None or limit >= len(sources))
    primary_beam = beam if full_beam else 1
    predictions = decode_sources(model, tok, cfg, sources, device, primary_beam, ev.get('beam_batch_size', 32) if full_beam else ev.get('batch_size', 64))
    score = score_predictions(predictions, sources, references)
    greedy_seconds = time.perf_counter() - begin
    metrics = dict(test_bleu=score['bleu'], test_chrf=score['chrf'], bleu_signature=score['bleu_signature'], chrf_signature=score['chrf_signature'], test_decoding='beam' if full_beam else 'greedy', test_beam_size=primary_beam, training_seconds=training_seconds, test_sources=len(sources), test_pairs=len(rows), primary_evaluation_seconds=greedy_seconds, config=cfg, data_sha256=data_hash, device=str(device), purpose='cpu_smoke_only' if cfg['data'].get('max_train') else 'full_training')
    metrics.update(test_identity(rows, data_hash, cfg['seed']))
    metrics.update(model_family='scratch_transformer', stage='trained', no_repeat_ngram_size=0)
    if full_beam:
        metrics.update(test_beam_bleu=score['bleu'], test_beam_chrf=score['chrf'])
    else:
        metrics.update(test_greedy_bleu=score['bleu'], test_greedy_chrf=score['chrf'])
    if full_beam and ev.get('report_greedy', False):
        greedy = decode_sources(model, tok, cfg, sources, device, 1)
        greedy_score = score_predictions(greedy, sources, references)
        metrics.update(test_greedy_bleu=greedy_score['bleu'], test_greedy_chrf=greedy_score['chrf'])
    if ev.get('repetition_ablation_ngram', 0):
        n = ev['repetition_ablation_ngram']
        constrained = decode_sources(model, tok, cfg, sources, device, beam, ev.get('beam_batch_size', 32), no_repeat_ngram_size=n)
        constrained_score = score_predictions(constrained, sources, references)
        metrics['repetition_ablation'] = dict(no_repeat_ngram_size=n, beam_size=beam, **constrained_score, sources=len(sources))
        with (output / 'samples_no_repeat.txt').open('w', encoding='utf-8') as f:
            for source, hyp in list(zip(sources, constrained))[:30]:
                f.write(f"EN: {source}\nREF: {' || '.join(references[source])}\nMODEL: {hyp}\n\n")
    if beam > 1 and (not full_beam) and limit:
        subset = sorted(sources, key=lambda s: hashlib.sha256(s.encode()).hexdigest())[:limit]
        started = time.perf_counter()
        beam_pred = decode_sources(model, tok, cfg, subset, device, beam, ev.get('beam_batch_size', 32))
        beam_score = score_predictions(beam_pred, subset, references)
        by_source = dict(zip(sources, predictions))
        subset_greedy = score_predictions([by_source[s] for s in subset], subset, references)
        metrics.update(beam_subset_sources=len(subset), beam_subset_size=beam, beam_subset_bleu=beam_score['bleu'], beam_subset_chrf=beam_score['chrf'], beam_subset_bleu_signature=beam_score['bleu_signature'], beam_subset_chrf_signature=beam_score['chrf_signature'], beam_subset_greedy_bleu=subset_greedy['bleu'], beam_subset_greedy_chrf=subset_greedy['chrf'], beam_subset_seconds=time.perf_counter() - started)
    metrics['evaluation_seconds'] = time.perf_counter() - begin
    write_artifacts(output, metrics, sources, references, predictions)
    return metrics

def checkpoint_training_metadata(state, output):
    """Preserve completed-run metadata when re-evaluating its best-only artifact."""
    cfg = state['config']
    metadata = dict(experiment=cfg.get('experiment', ''), training_status='checkpoint_evaluation', best_epoch=state.get('best_epoch'), completed_epochs=state.get('completed_epochs', len(state.get('history', []))))
    metadata.update({k: state[k] for k in ('run_id', 'training_seconds') if k in state})
    metadata.update(state.get('training_metadata', {}))
    previous = Path(output) / 'metrics.json'
    if previous.exists() and state.get('run_id'):
        old = json.loads(previous.read_text())
        if old.get('run_id') == state['run_id'] and old.get('best_epoch') == state.get('best_epoch'):
            metadata.update({k: old[k] for k in ('training_status', 'stop_reason', 'best_epoch', 'completed_epochs', 'training_seconds', 'attempted_steps', 'optimizer_steps', 'model_variant') if k in old})
    return metadata

## 7. Stable optimization, live progress and best-only saving
    Label smoothing, Adam/Noam, clipping, FP16 GradScaler, per-step LR/gradient diagnostics, epoch timing
    and ETA are embedded below. Nonfinite losses/gradients and sharp validation regressions stop a run.
    The original learning loop is preserved; the notebook saver writes **only** compact `best.pt`, never `last.pt`.
    It saves selected weights and tokenizer/config metadata, without optimizer/RNG state. CLI resume is intentionally unavailable here.

In [ ]:
"""Auditable optimizer updates and bounded divergence guards (also usable in CPU tests)."""
import csv
import json
import math
from pathlib import Path
import torch

class DivergenceError(RuntimeError):
    pass

def validation_diverged(value, best, settings):
    if not math.isfinite(value):
        return True
    return math.isfinite(best) and value > best * settings.get('val_ratio', 1.25) and (value - best >= settings.get('val_delta', 0.5))

def optimizer_update(model, optimizer, scheduler, scaler, clip):
    """Unscale before clipping; never step optimizer or scheduler on nonfinite norm."""
    scale_before = scaler.get_scale()
    scaler.unscale_(optimizer)
    norm = torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
    finite = bool(torch.isfinite(norm))
    proposed_lr = scheduler.rate(scheduler.steps + 1)
    if finite:
        lr_used = scheduler.step()
        scaler.step(optimizer)
        scaler.update()
    else:
        lr_used = None
        optimizer.zero_grad(set_to_none=True)
        if scaler.is_enabled():
            scaler.update(new_scale=scale_before * 0.5)
    return dict(grad_norm=float(norm), grad_norm_clipped=min(float(norm), clip) if finite else None, proposed_lr=proposed_lr, lr_used=lr_used, optimizer_step=scheduler.steps, updated=finite, scale_before=scale_before, scale_after=scaler.get_scale())

class StepLog:
    fields = ['epoch', 'batch', 'attempt_step', 'optimizer_step', 'loss', 'grad_norm', 'grad_norm_clipped', 'proposed_lr', 'lr_used', 'updated', 'scale_before', 'scale_after', 'event']

    def __init__(self, path, restored_rows=0):
        self.path = Path(path)
        self.count = restored_rows
        if self.path.exists():
            with self.path.open() as f:
                rows = list(csv.DictReader(f))[:restored_rows]
            with self.path.open('w', newline='') as f:
                w = csv.DictWriter(f, fieldnames=self.fields)
                w.writeheader()
                w.writerows(rows)
        else:
            self.path.parent.mkdir(parents=True, exist_ok=True)
            with self.path.open('w', newline='') as f:
                csv.DictWriter(f, fieldnames=self.fields).writeheader()
        self.file = self.path.open('a', newline='')
        self.writer = csv.DictWriter(self.file, fieldnames=self.fields)

    def write(self, epoch, batch, loss, update=None, event=''):
        self.count += 1
        row = dict(epoch=epoch, batch=batch, attempt_step=self.count, loss=loss, event=event)
        if update:
            row.update(update)
        self.writer.writerow(row)
        self.file.flush()

    def close(self):
        self.file.close()

def save_status(output, status, **details):
    path = Path(output) / 'run_status.json'
    path.write_text(json.dumps(dict(status=status, **details), indent=2))

"""Token-bucket training with measured progress, AMP checks and epoch-boundary resume."""
import argparse
import csv
import hashlib
import json
from pathlib import Path
import random
import time
import uuid
import numpy as np
import torch
from tqdm.auto import tqdm
import yaml

def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def sync(device):
    if device.type == 'cuda':
        torch.cuda.synchronize(device)

class Noam:

    def __init__(self, optimizer, d_model, warmup, factor=1.0):
        self.optimizer, self.d_model, self.warmup, self.factor = (optimizer, d_model, warmup, factor)
        self.steps = 0

    def rate(self, step):
        return self.factor * self.d_model ** (-0.5) * min(step ** (-0.5), step * self.warmup ** (-1.5))

    def step(self):
        self.steps += 1
        lr = self.rate(self.steps)
        for group in self.optimizer.param_groups:
            group['lr'] = lr
        return lr

def run_epoch(model, loader, criterion, device, optimizer=None, scheduler=None, scaler=None, clip=1.0, stats=None, progress=False, description='epoch', mixed_precision=None, log_every=25, step_log=None, epoch=0, divergence=None):
    training = optimizer is not None
    model.train(training)
    guard = divergence or {}
    amp_enabled = device.type == 'cuda' and (scaler is not None and scaler.is_enabled() if mixed_precision is None else mixed_precision)
    loss_sum = torch.zeros((), device=device)
    count, padded, all_tokens, skipped, consecutive = (0, 0, 0, 0, 0)
    first_dtype = None
    sync(device)
    begin = time.perf_counter()
    step = 0
    bar = tqdm(loader, desc=description, disable=not progress, mininterval=1, unit='step')
    try:
        with torch.set_grad_enabled(training):
            for step, (src, tgt) in enumerate(bar, 1):
                tokens = (tgt[:, 1:] != PAD).sum().item()
                all_tokens += (src != PAD).sum().item() + (tgt != PAD).sum().item()
                padded += src.numel() + tgt.numel()
                src, tgt = (src.to(device, non_blocking=True), tgt.to(device, non_blocking=True))
                if training:
                    optimizer.zero_grad(set_to_none=True)
                with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp_enabled):
                    logits = model(src, tgt[:, :-1])
                    loss = criterion(logits.reshape(-1, logits.size(-1)), tgt[:, 1:].reshape(-1))
                if first_dtype is None:
                    first_dtype = str(logits.dtype)
                    if amp_enabled and logits.dtype != torch.float16:
                        raise RuntimeError(f'FP16 AMP requested but logits dtype is {logits.dtype}')
                    if progress:
                        tqdm.write(f'{description}: autocast={amp_enabled}, logits={logits.dtype}, GradScaler={(scaler.is_enabled() if scaler is not None else False)}')
                if not torch.isfinite(loss):
                    if step_log and training:
                        step_log.write(epoch, step, str(loss.item()), event='nonfinite_loss')
                    raise DivergenceError(f'{description} batch {step}: nonfinite loss {loss.item()}')
                if training:
                    scaler.scale(loss).backward()
                    update = optimizer_update(model, optimizer, scheduler, scaler, clip)
                    if step_log:
                        step_log.write(epoch, step, loss.item(), update, '' if update['updated'] else 'nonfinite_gradient_update_skipped')
                    if not update['updated']:
                        skipped += 1
                        consecutive += 1
                        print(f"WARNING: {description} batch {step}: nonfinite unscaled gradient norm; optimizer/Noam skipped; scale {update['scale_before']} -> {update['scale_after']}", flush=True)
                        if not scaler.is_enabled() or consecutive >= guard.get('max_consecutive_skips', 8):
                            raise DivergenceError('Nonfinite gradients or repeated AMP overflow')
                        if step >= guard.get('skip_check_steps', 20) and skipped / step > guard.get('max_skip_fraction', 0.25):
                            raise DivergenceError('Too many skipped AMP updates')
                    else:
                        consecutive = 0
                loss_sum += loss.detach().float() * tokens
                count += tokens
                if progress and (step == 1 or step % log_every == 0 or step == len(loader)):
                    bar.set_postfix(loss=f'{loss_sum.item() / count:.3f}', target_tok_s=f'{count / (time.perf_counter() - begin):.0f}', lr=f"{optimizer.param_groups[0]['lr']:.2g}" if training else '-', skipped=skipped)
        if not count:
            raise ValueError('Empty epoch')
        if training and skipped == step:
            raise DivergenceError('No successful optimizer updates in this epoch')
        if not torch.isfinite(loss_sum):
            raise DivergenceError('Nonfinite epoch loss accumulation')
        return loss_sum.item() / count
    finally:
        bar.close()
        sync(device)
        seconds = time.perf_counter() - begin
        if stats is not None:
            stats.update(seconds=seconds, target_tokens=count, target_tokens_per_second=count / max(seconds, 1e-09), nonpad_tokens=all_tokens, nonpad_tokens_per_second=all_tokens / max(seconds, 1e-09), padding_fraction=1 - all_tokens / padded if padded else 0.0, steps=step, skipped_steps=skipped, amp_enabled=amp_enabled, logits_dtype=first_dtype)

def save_checkpoint(path, state):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix('.tmp')
    torch.save(state, temporary)
    temporary.replace(path)

def runtime_estimate(history, cfg, test_sources):
    recent = history[-min(3, len(history)):]
    per_epoch = sum((r['epoch_seconds'] + r.get('checkpoint_seconds', 0) for r in recent)) / len(recent)
    greedy_sources = sum((r.get('val_greedy_sources', 0) for r in recent))
    greedy_seconds = sum((r.get('val_greedy_seconds', 0.0) for r in recent))
    ev = cfg['evaluation']
    rate = greedy_sources / greedy_seconds if greedy_seconds > 0 else None
    limit = ev.get('beam_max_sources')
    beam_sources = min(test_sources, test_sources if limit is None else limit) if ev['beam_size'] > 1 else 0
    full_beam = ev['beam_size'] > 1 and (limit is None or limit >= test_sources)
    equivalent_greedy_sources = test_sources * ev['beam_size'] if full_beam else test_sources + beam_sources * ev['beam_size']
    if full_beam and ev.get('report_greedy', False):
        equivalent_greedy_sources += test_sources
    if ev.get('repetition_ablation_ngram', 0):
        equivalent_greedy_sources += test_sources * ev['beam_size']
    decode_seconds = equivalent_greedy_sources / rate if rate else None
    remaining_epochs = max(0, cfg['training']['epochs'] - history[-1]['epoch'])
    total = per_epoch * cfg['training']['epochs'] + (decode_seconds or 0)
    return dict(measured_epoch_seconds=per_epoch, completed_epoch=history[-1]['epoch'], planned_epochs=cfg['training']['epochs'], remaining_train_seconds=per_epoch * remaining_epochs, projected_total_hours=total / 3600, projected_test_seconds=decode_seconds, greedy_sources_per_second=rate, beam_time_assumption='beam_size times measured validation greedy cost per source', note='Estimate excludes one-time setup; data distribution, decoding lengths and Drive speed can change it. Early stopping can shorten it.', within_3_hour_target=total <= 3 * 3600)

def scratch_run(config_path, resume=None, checkpoint_dir=None, results_dir=None, stop_after_epochs=None, skip_test=False):
    cfg = yaml.safe_load(Path(config_path).read_text())
    if checkpoint_dir:
        cfg['paths']['checkpoints'] = checkpoint_dir
    if results_dir:
        cfg['paths']['results'] = results_dir
    seed_all(cfg['seed'])
    training = cfg['training']
    torch.set_num_threads(training.get('cpu_threads', 2))
    device = torch.device('cuda' if torch.cuda.is_available() and training['device'] != 'cpu' else 'cpu')
    if training['device'] == 'cuda' and device.type != 'cuda':
        raise RuntimeError('CUDA requested but unavailable; use a CPU config for smoke tests')
    paths = cfg['paths']
    checkpoint_root, output = (Path(paths['checkpoints']), Path(paths['results']))
    checkpoint_root.mkdir(parents=True, exist_ok=True)
    output.mkdir(parents=True, exist_ok=True)
    if not resume and (any(((checkpoint_root / name).exists() for name in ('last.pt', 'best.pt'))) or (output / 'run_status.json').exists()):
        raise FileExistsError('Existing experiment: use --resume or a new --checkpoint-dir')
    data_hash = hashlib.sha256(Path(paths['raw']).read_bytes()).hexdigest()
    state = torch.load(resume, map_location='cpu', weights_only=False) if resume else None
    if state:
        if not state.get('resume_supported', True):
            raise ValueError('Best-only inference checkpoint cannot resume training; start a new run')
        status_file = output / 'run_status.json'
        if status_file.exists() and json.loads(status_file.read_text())['status'] in ('diverged', 'pruned'):
            raise ValueError('This run diverged or was pruned; start a new experiment')
        old = state['config']
        for key in ('seed', 'model', 'tokenizer', 'data'):
            if old[key] != cfg[key]:
                raise ValueError(f'Cannot resume with changed {key}; start a new fast experiment')
        runtime_keys = ('epochs', 'device', 'cpu_threads', 'num_workers', 'pin_memory', 'prefetch_factor', 'progress', 'log_every')
        for key in training:
            if key not in runtime_keys and old['training'].get(key) != training[key]:
                raise ValueError(f'Cannot resume with changed training.{key}')
        if state.get('pipeline_version', 1) != 3:
            raise ValueError('This batching/RNG pipeline changed; start a new run instead of resuming legacy checkpoints')
        if state['data_sha256'] != data_hash:
            raise ValueError('Raw data changed since checkpoint')
    run_id = state['run_id'] if state else uuid.uuid4().hex
    save_status(output, 'running', run_id=run_id)
    print(f"Preparing {paths['raw']}", flush=True)
    rows = selected_splits(cfg)
    if state:
        tokenizer_path = checkpoint_root / 'tokenizer.model'
        tokenizer_path.write_bytes(state['tokenizer_model'])
        tok = Tokenizer(tokenizer_path)
    else:
        print('Fitting train-only SentencePiece', flush=True)
        tok = train_tokenizer(rows['train'], checkpoint_root / 'tokenizer', cfg['tokenizer']['vocab_size'])
    tokenizer_bytes = (checkpoint_root / 'tokenizer.model').read_bytes()
    model = Transformer(len(tok), **cfg['model']).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0, betas=(0.9, 0.98), eps=1e-09)
    scheduler = Noam(optimizer, cfg['model']['d_model'], training['warmup_steps'], training.get('lr_factor', 1.0))
    scaler = torch.amp.GradScaler('cuda', enabled=device.type == 'cuda' and training['mixed_precision'])
    criterion = torch.nn.CrossEntropyLoss(ignore_index=PAD, label_smoothing=training['label_smoothing'])
    start, best, bad, history, elapsed = (0, float('inf'), 0, [], 0.0)
    best_weights, best_epoch = (None, 0)
    if state:
        model.load_state_dict(state['model'])
        optimizer.load_state_dict(state['optimizer'])
        scaler.load_state_dict(state['scaler'])
        scheduler.steps = state['scheduler_steps']
        start, best, bad = (state['epoch'], state['best_loss'], state['bad_epochs'])
        history, elapsed, best_weights = (state['history'], state['training_seconds'], state['best_model'])
        best_epoch = state['best_epoch']
        random.setstate(state['rng_python'])
        np.random.set_state(state['rng_numpy'])
        torch.set_rng_state(state['rng_torch'])
        if device.type == 'cuda' and state['rng_cuda'] is not None:
            torch.cuda.set_rng_state_all(state['rng_cuda'])
    train_loader = make_loader(rows['train'], tok, cfg, True)
    val_loader = make_loader(rows['val'], tok, cfg, False)
    test_sources = len({r['en'] for r in rows['test']})
    print(json.dumps(dict(device=str(device), gpu=torch.cuda.get_device_name(0) if device.type == 'cuda' else None, amp_requested=training['mixed_precision'], amp_enabled=scaler.is_enabled(), parameters=sum((p.numel() for p in model.parameters())), num_workers=train_loader.num_workers, pin_memory=train_loader.pin_memory, train_batches=loader_stats(train_loader)), indent=2), flush=True)
    log_path = output / 'train_log.csv'

    def write_log():
        columns = ['epoch', 'train_loss', 'val_loss', 'lr', 'epoch_seconds', 'checkpoint_seconds', 'train_seconds', 'val_seconds', 'target_tokens_per_second', 'nonpad_tokens_per_second', 'padding_fraction', 'val_greedy_bleu', 'val_greedy_chrf', 'val_greedy_seconds', 'val_greedy_sources', 'cuda_peak_gb', 'amp_enabled', 'logits_dtype', 'skipped_steps']
        with log_path.open('w', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=columns)
            writer.writeheader()
            writer.writerows(history)
    write_log()
    step_log = StepLog(output / 'step_log.csv', state.get('step_log_rows', 0) if state else 0)
    training_status, stop_reason = ('completed', '')
    guard = training.get('divergence', {})
    limit_epoch = min(training['epochs'], start + stop_after_epochs) if stop_after_epochs else training['epochs']
    for epoch in range(start, limit_epoch):
        if bad >= training['patience']:
            training_status, stop_reason = ('early_stopped', 'validation patience exhausted')
            break
        set_loader_epoch(train_loader, epoch, cfg['seed'])
        if device.type == 'cuda':
            torch.cuda.reset_peak_memory_stats(device)
        begin = time.perf_counter()
        train_stats, val_stats = ({}, {})
        common = dict(progress=training.get('progress', True), mixed_precision=scaler.is_enabled(), log_every=training.get('log_every', 25))
        try:
            train_loss = run_epoch(model, train_loader, criterion, device, optimizer, scheduler, scaler, training['gradient_clip'], stats=train_stats, step_log=step_log, epoch=epoch + 1, divergence=guard, description=f"train {epoch + 1}/{training['epochs']}", **common)
            val_loss = run_epoch(model, val_loader, criterion, device, stats=val_stats, description=f"val {epoch + 1}/{training['epochs']}", **common)
        except DivergenceError as error:
            elapsed += time.perf_counter() - begin
            training_status, stop_reason = ('diverged', str(error))
            print(f'WARNING: divergence detected; stopping before another update: {stop_reason}', flush=True)
            (output / 'divergence.json').write_text(json.dumps(dict(epoch=epoch + 1, reason=stop_reason, optimizer_steps=scheduler.steps, attempted_steps=step_log.count, best_epoch=best_epoch), indent=2))
            break
        if validation_diverged(val_loss, best, guard):
            training_status = 'diverged'
            stop_reason = f'val_loss {val_loss:.4f} rose sharply above best {best:.4f}'
            print(f'WARNING: {stop_reason}; stopping and retaining best weights', flush=True)
        val_greedy = validation_greedy(model, tok, cfg, rows['val'], device) if training_status != 'diverged' else dict(val_greedy_bleu=None, val_greedy_chrf=None, val_greedy_seconds=0.0, val_greedy_sources=0)
        improved = val_loss < best - training['min_delta']
        if improved:
            best, bad, best_epoch = (val_loss, 0, epoch + 1)
            best_weights = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
        duration = time.perf_counter() - begin
        elapsed += duration
        history.append(dict(epoch=epoch + 1, train_loss=train_loss, val_loss=val_loss, lr=optimizer.param_groups[0]['lr'], epoch_seconds=duration, checkpoint_seconds=0.0, train_seconds=train_stats['seconds'], val_seconds=val_stats['seconds'], target_tokens_per_second=train_stats['target_tokens_per_second'], nonpad_tokens_per_second=train_stats['nonpad_tokens_per_second'], padding_fraction=train_stats['padding_fraction'], **val_greedy, cuda_peak_gb=torch.cuda.max_memory_allocated(device) / 1024 ** 3 if device.type == 'cuda' else 0.0, amp_enabled=train_stats['amp_enabled'], logits_dtype=train_stats['logits_dtype'], skipped_steps=train_stats['skipped_steps']))
        if training_status == 'diverged':
            write_log()
            (output / 'divergence.json').write_text(json.dumps(dict(epoch=epoch + 1, reason=stop_reason, val_loss=val_loss, best_loss=best, best_epoch=best_epoch, optimizer_steps=scheduler.steps), indent=2))
            break
        state = dict(pipeline_version=3, run_id=run_id, best_epoch=best_epoch, step_log_rows=step_log.count, config=cfg, model=model.state_dict(), best_model=best_weights, optimizer=optimizer.state_dict(), scaler=scaler.state_dict(), scheduler_steps=scheduler.steps, epoch=epoch + 1, best_loss=best, bad_epochs=bad, history=history, training_seconds=elapsed, tokenizer_model=tokenizer_bytes, data_sha256=data_hash, rng_python=random.getstate(), rng_numpy=np.random.get_state(), rng_torch=torch.get_rng_state(), rng_cuda=torch.cuda.get_rng_state_all() if device.type == 'cuda' else None)
        checkpoint_begin = time.perf_counter()
        save_checkpoint(checkpoint_root / 'last.pt', state)
        if improved:
            save_checkpoint(checkpoint_root / 'best.pt', state)
        history[-1]['checkpoint_seconds'] = time.perf_counter() - checkpoint_begin
        write_log()
        estimate = runtime_estimate(history, cfg, test_sources)
        (output / 'runtime_estimate.json').write_text(json.dumps(estimate, indent=2))
        print(json.dumps(history[-1]), flush=True)
        print(f"Epoch {epoch + 1}: {duration:.1f}s + checkpoint {history[-1]['checkpoint_seconds']:.1f}s; train ETA {estimate['remaining_train_seconds'] / 60:.1f} min; projected train+test {estimate['projected_total_hours']:.2f} h", flush=True)
        if not estimate['within_3_hour_target']:
            print('WARNING: measured first/recent epochs exceed the 3h target; see runtime_estimate.json. A new smaller experiment or fewer planned epochs may be needed.', flush=True)
    step_log.close()
    if training_status == 'completed' and bad >= training['patience']:
        training_status, stop_reason = ('early_stopped', 'validation patience exhausted')
    save_status(output, training_status, run_id=run_id, reason=stop_reason, best_epoch=best_epoch, completed_epochs=len(history), evaluation_complete=False)
    if best_weights is None:
        raise DivergenceError('No finite completed epoch: see run_status.json, divergence.json and step_log.csv')
    if skip_test:
        if training_status == 'diverged':
            raise DivergenceError(stop_reason)
        save_status(output, 'calibrated', run_id=run_id, best_epoch=best_epoch, completed_epochs=len(history), evaluation_complete=False)
        print('Calibration complete; test skipped. Resume last.pt to continue the SAME run.', flush=True)
        return runtime_estimate(history, cfg, test_sources)
    model.load_state_dict(best_weights)
    metrics = evaluate_model(model, tok, cfg, rows['test'], output, device, elapsed, data_hash)
    metrics.update(experiment=cfg.get('experiment', ''), run_id=run_id, model_variant=f"d{cfg['model']['d_model']}-l{cfg['model']['layers']}-" + ('preLN' if cfg['model'].get('norm_first') else 'postLN'), training_status=training_status, stop_reason=stop_reason, best_epoch=best_epoch, completed_epochs=len(history), attempted_steps=step_log.count, optimizer_steps=scheduler.steps)
    (output / 'metrics.json').write_text(json.dumps(metrics, ensure_ascii=False, indent=2))
    save_status(output, training_status, run_id=run_id, reason=stop_reason, best_epoch=best_epoch, completed_epochs=len(history), evaluation_complete=True)
    print(json.dumps(metrics, ensure_ascii=False), flush=True)
    return metrics

def save_checkpoint(path, state):
    path = Path(path)
    if path.name != 'best.pt':
        return  # no last checkpoint, optimizer duplicates or resume state on Drive
    weights = dict(state['best_model'])
    if state['config']['model'].get('tie_embeddings'):
        weights['output.weight'] = weights['embedding.weight']
    compact = {k:state[k] for k in ('run_id','config','best_epoch','history','training_seconds','tokenizer_model','data_sha256')}
    compact.update(best_model=weights, artifact_type='inference_best', resume_supported=False, pipeline_version=3)
    path.parent.mkdir(parents=True,exist_ok=True)
    temporary = path.with_suffix('.tmp')
    torch.save(compact, temporary)
    temporary.replace(path)

def notebook_config(name):
    cfg=copy.deepcopy(ORIGINAL_CONFIGS[name])
    run_name={'fast':'scratch_fast_full_notebook','large':'scratch_large_full_notebook','helsinki':'helsinki_full_notebook'}[name]
    base=ROOT/run_name
    cfg['paths'].update(raw=str(RAW),processed=str(WORK/'processed'),checkpoints=str(base/'checkpoints'),results=str(base/'results'))
    cfg['training']['num_workers']=0
    if CPU_SMOKE:
        cfg['data'].update(max_train=48,max_val=8,max_test=40)
        cfg['training'].update(device='cpu',epochs=1,patience=3,mixed_precision=False,pin_memory=False,progress=False)
        cfg['evaluation'].update(batch_size=16,beam_batch_size=8)
        if name in ('fast','large'):
            # Preserve Post-LN vs Pre-LN/tied behavior, shrink dimensions only for smoke.
            cfg['model'].update(d_model=32,heads=4,layers=1,d_ff=64,max_length=16)
            cfg['tokenizer']['vocab_size']=256
            cfg['training'].update(max_tokens=256,max_sentences=8,bucket_size=32,warmup_steps=10,lr_factor=.2)
            cfg['evaluation']['val_greedy_samples']=4
        else:
            cfg['data']['max_length']=16
            cfg['training'].update(batch_size=8,eval_batch_size=8,gradient_accumulation_steps=1,logging_steps=1)
            cfg['evaluation']['max_new_tokens']=16
    return cfg

def train_scratch_notebook(name):
    cfg=notebook_config(name)
    path=WORK/(name+'.yaml'); path.write_text(yaml.safe_dump(cfg))
    metrics=scratch_run(path)
    if metrics.get('training_status')=='diverged':
        raise DivergenceError('Inspect step_log.csv and divergence.json; do not compare failed weights')
    return Path(cfg['paths']['results'])

## 8. Fast v4: optional training and full-test evaluation
    Default: reuse the completed fast v4 artifacts. Opt-in: 12 epochs, Post-LN d256/3 layers and the exact
    Noam/batch settings above. A new run name protects the final experiment. Progress streams directly in the cell.

In [ ]:
RESULT_PATHS={name:ROOT/run['path'] for name,run in zip(('fast','large','zero_shot','finetuned'),FINAL_REPORT['runs'])}
if RUN_FAST:
    RESULT_PATHS['fast']=train_scratch_notebook('fast')
else:
    print('Fast: reusing',RESULT_PATHS['fast'])

## 9. Stable large v4: optional training and full-test evaluation
    Default: reuse Drive results. Opt-in: Pre-LN d512/6 layers, tied embeddings, warmup 3,000, factor 0.4,
    16,384 tokens/batch, 30 epochs, patience 5. Large v3 collapsed at an excessive early LR peak; v4 converged.
    no-repeat-3 had no meaningful gain (59.88 → 59.87 BLEU); both policies are recorded when reproducing.

In [ ]:
if RUN_LARGE:
    RESULT_PATHS['large']=train_scratch_notebook('large')
else:
    print('Large: reusing',RESULT_PATHS['large'])

## 10. Marian datasets, cached generation and shared evaluation
    Helsinki uses its pretrained tokenizer, **the identical split and multi-reference scorer**, cached batched
    generation and full-test beam 4/greedy. OPUS may contain Tatoeba; pretraining contamination is unresolved.
    Recovered Helsinki metrics and samples are valid completed artifacts even though their original CSV logs are missing.

In [ ]:
"""Same-split Marian zero-shot and Seq2SeqTrainer fine-tuning. Run GPU learning in Colab."""
import argparse
from contextlib import nullcontext
import csv
import hashlib
import json
import math
from pathlib import Path
import time
import numpy as np
import torch
from torch.utils.data import Dataset
from tqdm.auto import tqdm
import yaml
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, DataCollatorForSeq2Seq, EarlyStoppingCallback, Seq2SeqTrainer, Seq2SeqTrainingArguments, TrainerCallback, set_seed
from transformers.trainer_utils import get_last_checkpoint
LOG_FIELDS = ['epoch', 'train_loss', 'val_loss', 'lr', 'epoch_seconds', 'global_step', 'cuda_peak_gb']

def write_log(path, rows):
    with Path(path).open('w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=LOG_FIELDS)
        writer.writeheader()
        writer.writerows(rows)

class ParallelDataset(Dataset):

    def __init__(self, rows, tokenizer, max_length):
        self.items = []
        for start in range(0, len(rows), 1024):
            batch = rows[start:start + 1024]
            encoded = tokenizer([r['en'] for r in batch], text_target=[r['ru'] for r in batch], max_length=max_length, truncation=True)
            self.items.extend(({k: v[i] for k, v in encoded.items()} for i in range(len(batch))))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        return self.items[i]

class MarianCollator:
    """Dynamic padding + CPU label shift, without pickling a CUDA model into workers."""

    def __init__(self, tokenizer, config, multiple=None):
        self.padding = DataCollatorForSeq2Seq(tokenizer, model=None, pad_to_multiple_of=multiple)
        self.pad, self.start = (config.pad_token_id, config.decoder_start_token_id)

    def __call__(self, features):
        batch = self.padding(features)
        labels = batch['labels']
        decoder = torch.full_like(labels, self.pad)
        decoder[:, 0] = self.start
        decoder[:, 1:] = labels[:, :-1]
        decoder.masked_fill_(decoder == -100, self.pad)
        batch['decoder_input_ids'] = decoder
        return batch

@torch.inference_mode()
def generate_texts(model, tokenizer, cfg, sources, device, beams):
    model.eval()
    ev = cfg['evaluation']
    size = ev['beam_batch_size'] if beams > 1 else ev['batch_size']
    order = sorted(range(len(sources)), key=lambda i: len(sources[i]))
    predictions = [None] * len(sources)
    amp = torch.autocast('cuda', dtype=torch.float16) if device.type == 'cuda' and cfg['training']['mixed_precision'] else nullcontext()
    with amp:
        for start in tqdm(range(0, len(order), size), desc=f'Marian beam={beams}', disable=not cfg['training']['progress'], mininterval=1):
            indices = order[start:start + size]
            inputs = tokenizer([sources[i] for i in indices], padding=True, truncation=True, max_length=cfg['data']['max_length'], return_tensors='pt').to(device)
            output = model.generate(**inputs, num_beams=beams, do_sample=False, max_new_tokens=ev['max_new_tokens'], length_penalty=ev['length_penalty'] if beams > 1 else 1.0, no_repeat_ngram_size=ev.get('no_repeat_ngram_size', 0), early_stopping=beams > 1, use_cache=True)
            for i, text in zip(indices, tokenizer.batch_decode(output, skip_special_tokens=True)):
                predictions[i] = text
    return predictions

def evaluate_helsinki(model, tokenizer, cfg, rows, output, stage, identity, revision, training_seconds):
    begin = time.perf_counter()
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    refs = reference_groups(rows)
    sources = list(refs)
    device = next(model.parameters()).device
    greedy = generate_texts(model, tokenizer, cfg, sources, device, 1)
    gs = score_predictions(greedy, sources, refs)
    beams = cfg['evaluation']['beam_size']
    predictions = generate_texts(model, tokenizer, cfg, sources, device, beams) if beams > 1 else greedy
    bs = score_predictions(predictions, sources, refs)
    metrics = dict(**identity, test_bleu=bs['bleu'], test_chrf=bs['chrf'], bleu_signature=bs['bleu_signature'], chrf_signature=bs['chrf_signature'], test_greedy_bleu=gs['bleu'], test_greedy_chrf=gs['chrf'], test_beam_bleu=bs['bleu'], test_beam_chrf=bs['chrf'], test_decoding='beam' if beams > 1 else 'greedy', test_beam_size=beams, model_family='marian_cpu_fixture' if revision.startswith('local:') else 'helsinki_marian', pretrained_model_validated=not revision.startswith('local:'), stage=stage, base_revision=revision, training_status='completed', no_repeat_ngram_size=cfg['evaluation'].get('no_repeat_ngram_size', 0), config=cfg, device=str(device), training_seconds=training_seconds, evaluation_seconds=time.perf_counter() - begin, purpose='cpu_smoke_only' if cfg['data'].get('max_train') else stage, pretraining_overlap_risk='OPUS may include Tatoeba; test overlap has not been ruled out')
    write_artifacts(output, metrics, sources, refs, predictions)
    if stage == 'zero_shot':
        write_log(output / 'train_log.csv', [])
    return metrics

class MarianTrainer(Seq2SeqTrainer):

    def get_train_dataloader(self):
        loader = super().get_train_dataloader()
        if self.args.dataloader_num_workers:
            getattr(loader, 'base_dataloader', loader).multiprocessing_context = 'spawn'
        return loader

    def get_eval_dataloader(self, eval_dataset=None):
        loader = super().get_eval_dataloader(eval_dataset)
        if self.args.dataloader_num_workers:
            getattr(loader, 'base_dataloader', loader).multiprocessing_context = 'spawn'
        return loader

class EpochLog(TrainerCallback):
    """Live epoch summaries and durable CSV/ETA across reconnects; Trainer owns optimizer/RNG state."""

    def __init__(self, output, planned_epochs, stop_after_epochs=None):
        self.output = Path(output)
        self.planned_epochs, self.stop_after = (planned_epochs, stop_after_epochs)
        self.rows = []
        path = self.output / 'train_log.csv'
        if path.exists():
            with path.open() as f:
                self.rows = [{k: float(v) if v else None for k, v in r.items()} for r in csv.DictReader(f)]

    def on_train_begin(self, args, state, control, **kwargs):
        self.start_epoch = state.epoch or 0
        self.rows = [r for r in self.rows if r['epoch'] <= self.start_epoch + 1e-06]
        write_log(self.output / 'train_log.csv', self.rows)

    def on_epoch_begin(self, args, state, control, **kwargs):
        self.begin = time.perf_counter()
        self.first_step = state.global_step
        if torch.cuda.is_available() and (not args.use_cpu):
            torch.cuda.reset_peak_memory_stats()

    def on_epoch_end(self, args, state, control, **kwargs):
        if self.stop_after and state.epoch >= self.start_epoch + self.stop_after - 1e-06:
            control.should_training_stop = True
        return control

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        elapsed = time.perf_counter() - self.begin
        records = [r for r in state.log_history if 'loss' in r and r.get('step', 0) > self.first_step]
        loss = float(np.mean([r['loss'] for r in records])) if records else None
        lr = records[-1].get('learning_rate') if records else None
        self.rows.append(dict(epoch=state.epoch, train_loss=loss, val_loss=metrics['eval_loss'], lr=lr, epoch_seconds=elapsed, global_step=state.global_step, cuda_peak_gb=torch.cuda.max_memory_allocated() / 1024 ** 3 if torch.cuda.is_available() and (not args.use_cpu) else 0.0))
        write_log(self.output / 'train_log.csv', self.rows)
        mean = sum((r['epoch_seconds'] for r in self.rows)) / len(self.rows)
        estimate = dict(measured_epoch_seconds=mean, planned_epochs=self.planned_epochs, completed_epoch=state.epoch, remaining_train_seconds=max(0, self.planned_epochs - state.epoch) * mean, projected_training_hours=mean * self.planned_epochs / 3600, note='Training+validation estimate; add zero-shot/final generation and checkpoint I/O. T4 target is 1–2 hours, not guaranteed.')
        (self.output / 'runtime_estimate.json').write_text(json.dumps(estimate, indent=2))
        print(f"\nMarian epoch {state.epoch:.2f}: {elapsed:.1f}s, val_loss={metrics['eval_loss']:.4f}, ETA={estimate['remaining_train_seconds'] / 60:.1f} min", flush=True)

### Best-only Marian storage and an offline CPU fixture
    `Seq2SeqTrainer` retains the same AdamW schedule, smoothing, batching, seeds and loss-based best selection.
    A callback saves only improvements into one `best/` directory (safetensors + tokenizer); it never writes
    epoch checkpoint directories or optimizer state. This trades training resume for Drive capacity.
    CPU validation builds tiny **random** Marian weights locally without downloading Helsinki; its results are explicitly labelled.

In [ ]:
from transformers import MarianConfig, MarianMTModel, MarianTokenizer
import shutil

class NotebookBest(TrainerCallback):
    def __init__(self,path,tokenizer,patience):
        self.path,self.tokenizer,self.patience=Path(path),tokenizer,patience
        self.best=float('inf'); self.bad=0
    def on_evaluate(self,args,state,control,metrics=None,model=None,**kwargs):
        value=metrics['eval_loss']
        if not math.isfinite(value):
            raise DivergenceError('Marian validation loss is nonfinite')
        if value < self.best:
            self.best=value; self.bad=0
            model.save_pretrained(self.path, safe_serialization=True)
            self.tokenizer.save_pretrained(self.path)
            (self.path/'best_selection.json').write_text(json.dumps(dict(epoch=state.epoch,val_loss=value)))
        else:
            self.bad+=1
            if self.bad >= self.patience:
                control.should_training_stop=True
        return control

def cpu_marian_fixture(cfg):
    root=WORK/'random_marian_fixture';root.mkdir(exist_ok=True)
    rows=selected_splits(cfg)['train']
    tok=train_tokenizer(rows,root/'sp',256)
    for name in ('source.spm','target.spm'):
        shutil.copyfile(root/'sp.model',root/name)
    (root/'vocab.json').write_text(json.dumps({tok.sp.id_to_piece(i):i for i in range(len(tok))},ensure_ascii=False))
    tokenizer=MarianTokenizer(vocab=str(root/'vocab.json'),source_spm=str(root/'source.spm'),target_spm=str(root/'target.spm'))
    tokenizer.save_pretrained(root)
    torch.manual_seed(42)
    config=MarianConfig(vocab_size=len(tokenizer),decoder_vocab_size=len(tokenizer),d_model=16,
        encoder_layers=1,decoder_layers=1,encoder_attention_heads=2,decoder_attention_heads=2,
        encoder_ffn_dim=32,decoder_ffn_dim=32,max_position_embeddings=64,pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,decoder_start_token_id=tokenizer.pad_token_id,forced_eos_token_id=None)
    model=MarianMTModel(config);model.generation_config.forced_eos_token_id=tokenizer.eos_token_id
    model.save_pretrained(root)
    cfg['model'].update(name=str(root),revision='main')
    return cfg

def setup_marian():
    cfg=notebook_config('helsinki')
    set_seed(cfg['seed']);torch.set_num_threads(cfg['training']['cpu_threads'])
    if CPU_SMOKE:
        cfg=cpu_marian_fixture(cfg)
    rows=selected_splits(cfg)
    device=torch.device('cpu' if CPU_SMOKE else 'cuda')
    model=AutoModelForSeq2SeqLM.from_pretrained(cfg['model']['name'],revision=cfg['model']['revision']).to(device)
    revision=getattr(model.config,'_commit_hash',None)
    if CPU_SMOKE:
        revision='local:'+hashlib.sha256((Path(cfg['model']['name'])/'model.safetensors').read_bytes()).hexdigest()
    elif not revision:
        raise RuntimeError('Cannot resolve pretrained model snapshot')
    tokenizer=AutoTokenizer.from_pretrained(cfg['model']['name'],revision=revision if not CPU_SMOKE else 'main',use_fast=False)
    identity=test_identity(rows['test'],hashlib.sha256(RAW.read_bytes()).hexdigest(),cfg['seed'])
    root=Path(cfg['paths']['checkpoints']); root.mkdir(parents=True,exist_ok=True)
    if (root/'best').exists():
        raise FileExistsError('Choose a new Helsinki reproduction run name before repeating')
    (root/'run_manifest.json').write_text(json.dumps(dict(config=cfg,identity=identity,resolved_revision=revision),indent=2))
    return cfg,rows,device,model,tokenizer,identity,revision

## 11. Helsinki zero-shot BEFORE any fine-tuning
    Default: load the reported zero-shot run. Enabling fine-tuning also evaluates its fresh base model first,
    even if the standalone zero-shot flag is off. Full-test greedy and beam4 are always recorded.

In [ ]:
MARIAN_CONTEXT=None
if RUN_HELSINKI_ZERO_SHOT or RUN_HELSINKI_FT:
    MARIAN_CONTEXT=setup_marian()
    hcfg,hrows,hdevice,hmodel,htok,hidentity,hrevision=MARIAN_CONTEXT
    output=Path(hcfg['paths']['results'])/'zero_shot'
    if (output/'metrics.json').exists():
        raise FileExistsError('Choose a fresh Helsinki reproduction results directory')
    evaluate_helsinki(hmodel,htok,hcfg,hrows['test'],output,'zero_shot',hidentity,hrevision,0.)
    RESULT_PATHS['zero_shot']=output
else:
    print('Zero-shot: reusing', RESULT_PATHS['zero_shot'])

## 12. Optional Helsinki fine-tuning with Seq2SeqTrainer
    Default: reuse the completed 3-epoch run. The reproduction uses loss-only validation, FP16 on T4,
    batch 64 × accumulation 2, LR 3e−5, 5% warmup and patience 2. Model generation is performed after
    restoring the validation-best model. Live Trainer progress and epoch CSV/ETA stay visible in this cell.
    No recovered historical training log is synthesized when this stage is skipped.

In [ ]:
if RUN_HELSINKI_FT:
    hcfg,hrows,hdevice,hmodel,htok,hidentity,hrevision=MARIAN_CONTEXT
    tr=hcfg['training']; output=Path(hcfg['paths']['results'])/'finetuned';output.mkdir(parents=True,exist_ok=True)
    root=Path(hcfg['paths']['checkpoints'])
    epoch_log=EpochLog(output,tr['epochs'])
    best=NotebookBest(root/'best',htok,tr['patience'])
    args=Seq2SeqTrainingArguments(output_dir=str(WORK/'trainer_scratch'),num_train_epochs=tr['epochs'],
        per_device_train_batch_size=tr['batch_size'],per_device_eval_batch_size=tr['eval_batch_size'],
        gradient_accumulation_steps=tr['gradient_accumulation_steps'],learning_rate=tr['learning_rate'],
        warmup_ratio=tr['warmup_ratio'],weight_decay=tr['weight_decay'],label_smoothing_factor=tr['label_smoothing'],
        max_grad_norm=tr['gradient_clip'],fp16=hdevice.type=='cuda' and tr['mixed_precision'],use_cpu=hdevice.type=='cpu',
        eval_strategy='epoch',save_strategy='no',logging_steps=tr['logging_steps'],logging_first_step=True,
        load_best_model_at_end=False,predict_with_generate=False,group_by_length=True,dataloader_num_workers=0,
        dataloader_pin_memory=tr['pin_memory'] and hdevice.type=='cuda',dataloader_persistent_workers=False,
        report_to=[],push_to_hub=False,seed=hcfg['seed'],data_seed=hcfg['seed'],disable_tqdm=not tr['progress'])
    trainer=Seq2SeqTrainer(model=hmodel,args=args,
        train_dataset=ParallelDataset(hrows['train'],htok,hcfg['data']['max_length']),
        eval_dataset=ParallelDataset(hrows['val'],htok,hcfg['data']['max_length']),processing_class=htok,
        data_collator=MarianCollator(htok,hmodel.config,8 if hdevice.type=='cuda' else None),callbacks=[epoch_log,best])
    trainer.train()
    hmodel=AutoModelForSeq2SeqLM.from_pretrained(root/'best').to(hdevice)
    elapsed=sum(r['epoch_seconds'] for r in epoch_log.rows)
    evaluate_helsinki(hmodel,htok,hcfg,hrows['test'],output,'fine_tuned',hidentity,hrevision,elapsed)
    RESULT_PATHS['finetuned']=output
    del trainer,hmodel
    MARIAN_CONTEXT=None
    if torch.cuda.is_available(): torch.cuda.empty_cache()
else:
    print('Fine-tuned Helsinki: reusing',RESULT_PATHS['finetuned'])

## 13. Exact-identity comparison and verbatim examples
    Missing metrics are reported rather than replaced with fabricated run artifacts. The separately labelled
    user-reported table remains available. Comparison refuses mismatched tests, metric signatures or source counts.
    It does not require Helsinki training CSVs. Standard sample files contain just 30 examples, so special error
    cases may be unavailable; an optional inference section follows.

In [ ]:
"""Build Markdown/CSV comparisons, rejecting known test mismatches and labeling legacy runs."""
import argparse
import csv
import json
from pathlib import Path

def result_status(metrics):
    """Old Helsinki writers emitted metrics only after evaluation, but omitted status.

    Recovered metrics/samples are sufficient; never require or manufacture a training
    log. Explicit statuses (including failures) always take precedence. Completion
    describes execution, not model quality or uncontaminated pretraining.
    """
    status = metrics.get('training_status')
    if status and status != 'unreported':
        return status
    if metrics.get('model_family') == 'helsinki_marian' and metrics.get('stage') in ('zero_shot', 'fine_tuned') and all((k in metrics for k in ('test_bleu', 'test_chrf', 'test_sources'))):
        return 'completed'
    return status or 'unreported'

def compare(paths, include_cpu=False, require_four=False):
    found = set()
    for path in paths:
        path = Path(path)
        if not path.exists():
            raise FileNotFoundError(path)
        if path.is_file():
            found.add(path)
        else:
            found.update(path.rglob('metrics.json'))
    metrics = [(p, json.loads(p.read_text())) for p in sorted(found)]
    metrics = [(p, m) for p, m in metrics if include_cpu or m.get('purpose') != 'cpu_smoke_only']
    if not metrics:
        raise ValueError('No result files (CPU smoke runs are excluded unless --include-cpu)')
    if require_four:
        roles = set()
        for _, m in metrics:
            cfg = m.get('config', {})
            family = m.get('model_family', '')
            experiment = m.get('experiment', cfg.get('experiment', ''))
            model = cfg.get('model', {})
            if family == 'scratch_transformer':
                if experiment.startswith('scratch_fast') or (model.get('d_model') == 256 and model.get('layers') == 3):
                    roles.add('fast')
                if experiment.startswith('scratch_large') or (model.get('d_model') == 512 and model.get('layers') == 6):
                    roles.add('large')
            elif family == 'helsinki_marian' or (include_cpu and family == 'marian_cpu_fixture'):
                if m.get('stage') in ('zero_shot', 'fine_tuned'):
                    roles.add(m['stage'])
        missing = {'fast', 'large', 'zero_shot', 'fine_tuned'} - roles
        if missing:
            raise ValueError(f'Missing required comparison runs: {sorted(missing)}')
        if not include_cpu:
            for path, m in metrics:
                if m.get('schema_version') != 2 or not m.get('test_split_sha256'):
                    raise ValueError(f'Required comparison needs schema v2 exact test identity: {path}')
                if m.get('test_decoding') != 'beam' or m.get('test_beam_size') != 4 or 'test_greedy_bleu' not in m or ('test_greedy_chrf' not in m):
                    raise ValueError(f'Required comparison needs full-test beam4 AND greedy: {path}')
    hashes = {m['test_split_sha256'] for _, m in metrics if m.get('test_split_sha256')}
    if len(hashes) > 1:
        raise ValueError('Different held-out reference sets: compare these in separate tables')
    missing_hash_counts = {m.get('config', {}).get('data', {}).get('max_test') for _, m in metrics}
    if len(missing_hash_counts) > 1 and any((not m.get('test_split_sha256') for _, m in metrics)):
        raise ValueError('Legacy runs with different test sampling cannot be verified')
    raw = {m['data_sha256'] for _, m in metrics if m.get('data_sha256')}
    seeds = {m.get('split_seed', m.get('config', {}).get('seed')) for _, m in metrics}
    seeds.discard(None)
    if len(raw) > 1 or len(seeds) > 1:
        raise ValueError('Different raw dataset or split seed')
    for field in ('bleu_signature', 'chrf_signature'):
        signatures = {m[field] for _, m in metrics if m.get(field)}
        if len(signatures) > 1:
            raise ValueError(f'Different metric signatures: {field}')
    counts = {m['test_sources'] for _, m in metrics}
    if len(counts) > 1:
        raise ValueError('Different test source counts')
    rows = []
    for path, m in metrics:
        identity = 'verified' if m.get('test_split_sha256') else 'legacy/unverified'
        policy = m.get('test_decoding', 'unknown')
        ngram = m.get('no_repeat_ngram_size', 0)
        if ngram:
            policy += f'/no-repeat-{ngram}'
        rows.append(dict(run=str(path.parent), model=m.get('experiment') or m.get('model_family', 'legacy'), status=result_status(m), stage=m.get('stage', 'legacy'), policy=policy, beam=m.get('test_beam_size', m.get('config', {}).get('evaluation', {}).get('beam_size', '?')), bleu=m['test_bleu'], chrf=m['test_chrf'], train_seconds=m['training_seconds'], eval_seconds=m.get('evaluation_seconds', 0.0), sources=m['test_sources'], identity=identity))
        if m.get('test_decoding') == 'beam' and 'test_greedy_bleu' in m:
            rows.append(dict(rows[-1], policy=f'greedy/no-repeat-{ngram}' if ngram else 'greedy', beam=1, bleu=m['test_greedy_bleu'], chrf=m['test_greedy_chrf']))
        if 'repetition_ablation' in m:
            a = m['repetition_ablation']
            rows.append(dict(rows[-1], policy=f"beam/no-repeat-{a['no_repeat_ngram_size']}", beam=a['beam_size'], bleu=a['bleu'], chrf=a['chrf']))
    return rows

def write_comparison(rows, output):
    output = Path(output)
    output.parent.mkdir(parents=True, exist_ok=True)
    fields = list(rows[0])
    with output.with_suffix('.csv').open('w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)
    lines = ['| Run | Model | Status | Stage | Decode | Beam | BLEU | chrF | Train s | Eval s | Sources | Identity |', '| --- | --- | --- | --- | --- | ---: | ---: | ---: | ---: | ---: | ---: | --- |']
    for r in rows:
        lines.append(f"| {r['run'].replace('|', '/')} | {r['model']} | {r['status']} | {r['stage']} | {r['policy']} | {r['beam']} | {r['bleu']:.2f} | {r['chrf']:.2f} | {r['train_seconds']:.1f} | {r['eval_seconds']:.1f} | {r['sources']} | {r['identity']} |")
    lines += ['', 'Times are per-run totals repeated per policy; do not sum rows. Full-test scores only. Legacy files lack exact reference-set verification. Legacy Helsinki evaluation artifacts imply completed execution even if the recovered training log is absent. Helsinki may have Tatoeba pretraining contamination. Different tokenizers/caps/decoding policies are not controlled ablations.']
    text = '\n'.join(lines) + '\n'
    output.with_suffix('.md').write_text(text)
    return text

"""Join exact model outputs by English source; missing outputs are never invented."""
import argparse
import json
from pathlib import Path
SOURCES = ['The plans were discarded.', 'Sometimes cows are killed by coyotes.', 'They have lost 10 games in a row since their winning streak ended.', 'She quickly put the money out of sight.', 'Tom is clearly lost.']

def read_samples(path):
    samples = {}
    if not Path(path).exists():
        return samples
    source = None
    for line in Path(path).read_text(encoding='utf-8').splitlines():
        if line.startswith('EN: '):
            source = line[4:]
        elif line.startswith('MODEL: ') and source is not None:
            samples[source] = line[7:]
    return samples

def example_table(root, summary='docs/final_results.json', sources=None, include_common=True):
    runs = json.loads(Path(summary).read_text())['runs']
    maps = [read_samples(Path(root) / r['path'] / 'samples.txt') for r in runs]
    chosen = list(SOURCES if sources is None else sources)
    if include_common and maps:
        common = set.intersection(*(set(m) for m in maps))
        chosen += sorted(common - set(chosen))[:10]

    def escaped(x):
        return x.replace('|', '\\|').replace('\n', ' ')
    lines = ['| English | ' + ' | '.join((r['name'] for r in runs)) + ' |', '| ' + ' | '.join(['---'] * (len(runs) + 1)) + ' |']
    for source in chosen:
        lines.append('| ' + ' | '.join((escaped(t) for t in [source] + [m.get(source, 'Exact output unavailable in samples.txt') for m in maps])) + ' |')
    lines += ['', 'Only verbatim saved outputs are shown. The standard samples.txt stores 30 sources; requested error cases may be absent. Missing translations require an existing prediction artifact or inference with saved best weights, not invented text.']
    return '\n'.join(lines) + '\n'

available=[p for p in RESULT_PATHS.values() if (p/'metrics.json').exists()]
print('User-reported final aggregate scores (not recomputed here):')
for report in FINAL_REPORT['runs']:
    print(report['name'], 'greedy:',report['greedy_bleu'],report['greedy_chrf'],
          'beam4:',report['beam_bleu'],report['beam_chrf'])
if available:
    table=compare(available,include_cpu=CPU_SMOKE,require_four=len(available)==4)
    print(write_comparison(table,ROOT/'full_notebook_comparison'))
for name,path in RESULT_PATHS.items():
    if not (path/'metrics.json').exists():
        print('Missing result:',name,path,'— check the Drive location; no training was started.')
    elif name=='finetuned' and not (path/'train_log.csv').exists():
        print('Helsinki recovered artifacts: completed; train_log.csv unavailable.')
sample_maps=[read_samples(RESULT_PATHS[name]/'samples.txt') for name in ('fast','large','zero_shot','finetuned')]
common=set.intersection(*(set(m) for m in sample_maps))
example_lines=['| English | Fast | Large v4 | Helsinki zero-shot | Helsinki fine-tuned |','| --- | --- | --- | --- | --- |']
for source in SOURCES+sorted(common-set(SOURCES))[:10]:
    values=[source]+[m.get(source,'Exact output unavailable') for m in sample_maps]
    example_lines.append('| '+' | '.join(v.replace('|',chr(92)+'|') for v in values)+' |')
examples='\n'.join(example_lines)+'\n'
(ROOT/'full_notebook_examples.md').write_text(examples,encoding='utf-8')
print(examples)

## 14. Optional exact four-model inference, with NO training
    Leave `RUN_EXAMPLE_INFERENCE=False` to avoid model downloads/loading. Enable only to fill missing examples.
    The paths below follow the original modular notebooks: scratch `checkpoints/best.pt`, Helsinki `checkpoints/final`.
    Adjust them if you moved saved weights; the inline reproduction stores Marian in `checkpoints/best` instead.
    Zero-shot must use the **original resolved base revision** recorded in its metrics; it is not inferred from current `main`.
    Only trusted local checkpoints should be loaded. Each model is freed before the next one to fit T4 memory.

In [ ]:
if RUN_EXAMPLE_INFERENCE:
    exact={}
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    for name,run_name in [('fast','scratch_fast_v4_fullbeam'),('large','scratch_large_stable_v4')]:
        model,tok,cfg,state,_=load_checkpoint(ROOT/run_name/'checkpoints/best.pt')
        cfg['training']['progress']=True
        exact[name]=decode_sources(model,tok,cfg,SOURCES,next(model.parameters()).device,4)
        del model,state
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    base_metrics=json.loads((RESULT_PATHS['zero_shot']/'metrics.json').read_text())
    hcfg=base_metrics['config']; revision=base_metrics['base_revision']
    for name,location in [('zero_shot',hcfg['model']['name']),('finetuned',str(ROOT/'helsinki_v1_seed42/checkpoints/final'))]:
        kwargs={'revision':revision} if name=='zero_shot' else {}
        model=AutoModelForSeq2SeqLM.from_pretrained(location,**kwargs).to(device)
        tok=AutoTokenizer.from_pretrained(location,use_fast=False,**kwargs)
        exact[name]=generate_texts(model,tok,hcfg,SOURCES,device,4)
        del model
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    lines=['| English | Fast | Large v4 | Helsinki zero-shot | Helsinki fine-tuned |','| --- | --- | --- | --- | --- |']
    for i,source in enumerate(SOURCES):
        lines.append('| '+' | '.join(v.replace('|',chr(92)+'|') for v in [source]+[exact[k][i] for k in ('fast','large','zero_shot','finetuned')])+' |')
    (ROOT/'exact_error_examples.md').write_text('\n'.join(lines)+'\n',encoding='utf-8')
    print('\n'.join(lines))
else:
    print('Saved-weight example inference skipped. No training or model download is needed for the existing results table.')

## 15. Export plots from REAL artifacts
    Large's curve requires its real epoch log. Missing Helsinki logs are expected and do not prevent plotting.
    Images go to Drive `full_notebook_figures`; copy them to repository `docs/figures/` and commit.
    The score chart is generated only when all four actual metrics are present. CPU fixtures are clearly titled.
    All notebook outputs must be cleared before committing.

In [ ]:
"""Export publication figures from real logs and metrics, or reported aggregate scores.

The aggregate JSON is explicitly labelled; epoch curves always require a real CSV.
"""
import argparse
import csv
import json
from pathlib import Path

def plot_results(summary, loss_log, output='docs/figures', results_root=None):
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    import numpy as np
    summary = json.loads(Path(summary).read_text(encoding='utf-8'))
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    runs = summary['runs']
    names = [r['name'].replace('Transformer ', 'Scratch ').replace('Helsinki ', 'Helsinki\n') for r in runs]
    bars = []
    for r in runs:
        if results_root:
            path = Path(results_root) / r['path'] / 'metrics.json'
            m = json.loads(path.read_text(encoding='utf-8'))
            if m['test_sources'] != summary['test_sources'] or m['test_split_sha256'] != summary.get('test_split_sha256', m['test_split_sha256']) or (not m['test_split_sha256'].startswith(summary['test_split_sha256_prefix'])):
                raise ValueError(f'Different held-out set: {path}')
            if m.get('test_decoding') != 'beam' or m.get('test_beam_size') != 4:
                raise ValueError(f'Expected full-test beam4: {path}')
            bars.append((m['test_greedy_bleu'], m['test_bleu'], m['test_greedy_chrf'], m['test_chrf']))
        else:
            bars.append((r['greedy_bleu'], r['beam_bleu'], r['greedy_chrf'], r['beam_chrf']))
    fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
    x = np.arange(len(runs))
    width = 0.36
    for ax, title, offset in zip(axes, ('sacreBLEU (13a)', 'chrF2'), (0, 2)):
        for k, label in enumerate(('Greedy', 'Beam 4')):
            values = [b[offset + k] for b in bars]
            rectangles = ax.bar(x + (k - 0.5) * width, values, width, label=label)
            ax.bar_label(rectangles, fmt='%.2f', fontsize=8)
        ax.set_xticks(x, names, fontsize=8)
        ax.set_ylim(0, 85)
        ax.set_ylabel(title)
        ax.legend()
        ax.spines[['top', 'right']].set_visible(False)
    fig.suptitle(f"EN→RU · {summary['test_sources']:,} held-out sources · multiple references")
    fig.savefig(output / 'model_scores.png', dpi=180)
    plt.close(fig)
    if loss_log:
        with Path(loss_log).open(encoding='utf-8', newline='') as f:
            rows = list(csv.DictReader(f))
        if not rows:
            raise ValueError('Empty loss CSV; no curve can be reconstructed')
        fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
        for key, label in [('train_loss', 'Train'), ('val_loss', 'Validation')]:
            ax.plot([float(r['epoch']) for r in rows], [float(r[key]) for r in rows], label=label)
        ax.set(xlabel='Epoch', ylabel='Label-smoothed cross-entropy', title='Transformer large v4: validation loss converged')
        ax.legend()
        ax.spines[['top', 'right']].set_visible(False)
        fig.savefig(output / 'large_v4_loss.png', dpi=180)
        plt.close(fig)
    provenance = {'scores_source': 'Drive metrics.json' if results_root else summary['provenance'], 'loss_source': str(loss_log) if loss_log else None, 'note': 'No missing epoch values have been interpolated or fabricated.'}
    (output / 'provenance.json').write_text(json.dumps(provenance, indent=2) + '\n')
    return output

summary=copy.deepcopy(FINAL_REPORT)
for report,key in zip(summary['runs'],('fast','large','zero_shot','finetuned')):
    report['path']=str(RESULT_PATHS[key])
if CPU_SMOKE and available:
    m=json.loads((available[0]/'metrics.json').read_text())
    summary.update(test_sources=m['test_sources'],test_split_sha256_prefix=m['test_split_sha256'],test_split_sha256=m['test_split_sha256'])
    for report in summary['runs']:report['name']='CPU fixture: '+report['name']
summary_path=WORK/'figure_summary.json';summary_path.write_text(json.dumps(summary,ensure_ascii=False))
log=RESULT_PATHS['large']/'train_log.csv'
if len(available)==4:
    print(plot_results(summary_path,log if log.exists() else None,ROOT/'full_notebook_figures',ROOT))
else:
    print('Figures skipped: full original artifacts unavailable. No epoch curve was fabricated.')